# 美股基本面與估值版，資料來源维持 Yahoo。
# 每批輸出 reports/詳細版/US 與 reports/簡化版/US 各一份 Excel。
# 簡化版47欄，財務安全性評價後新增五分類分析師共識及最低／最高目標價；現金流品質與分析師依據保留於詳細版。
# 不啟用技術／資金流分析，歷史彙整改為手動分版本執行。
# 無資料的即時EPS／即時本益比／即時估值位階不輸出；買價建議只依歷史估值百分位。


In [24]:
# 設立&基本組態
# =========================
# Setup & config
# =========================
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import date
from typing import Any, Iterable
import logging
import math
import pandas as pd
import yaml

LOGGER = logging.getLogger(__name__)

# =========================
# Config
# =========================
def load_config(path: str | Path | None = None):

    config_path = (
        Path(path)
        if path
        else Path("config/US.yaml")
    )


    with config_path.open(
        encoding="utf-8"
    ) as file:

        config = yaml.safe_load(file)


    # 加入設定檔名稱
    config["_config_name"] = (
        config_path.stem
    )
    return config


In [25]:
# 共同應用的函式
# =========================
# Shared helpers
# =========================
def safe_divide(numerator: float | None, denominator: float | None) -> float | None:
    if numerator is None or denominator in (None, 0):
        return None
    result = numerator / denominator
    return result if math.isfinite(result) else None


def percentile_rank(value: float | None, values: Iterable[float]) -> float | None:
    cleaned = sorted(item for item in values if item is not None and math.isfinite(item))
    if value is None or not cleaned:
        return None
    return 100 * sum(item <= value for item in cleaned) / len(cleaned)

In [26]:
# 函式定義
# =========================
# Data Models
# =========================

# =========================
# Core / Profile Models
# =========================
@dataclass
class Classification:
    market: str
    sector: str

# =========================
# Financial Data Models
# =========================
@dataclass
class FinancialSnapshot:
    ticker: str
    source: str
    as_of: date
    asset_type: str | None = None
    current_price: float | None = None
    week_52_high: float | None = None
    week_52_low: float | None = None
    market_cap: float | None = None
    enterprise_value: float | None = None
    revenue: float | None = None
    gross_profit: float | None = None
    operating_income: float | None = None
    net_income: float | None = None
    eps: float | None = None
    ttm_eps: float | None = None
    eps_ttm_nowcast: float | None = None
    pe_ttm_nowcast: float | None = None
    forward_eps: float | None = None
    book_value: float | None = None
    book_value_per_share: float | None = None
    free_cash_flow: float | None = None
    operating_cash_flow: float | None = None
    capex: float | None = None
    total_debt: float | None = None
    cash: float | None = None
    shares_outstanding: float | None = None
    average_shares_12m: float | None = None
    total_equity: float | None = None
    current_assets: float | None = None
    current_liabilities: float | None = None
    inventory: float | None = None
    interest_expense: float | None = None
    financial_history: dict[str, list[float]] = field(default_factory=dict)
    quarterly_reference: dict[str, Any] = field(default_factory=dict)
    cashflow_quality: dict[str, Any] = field(default_factory=dict)
    fundamental_review_inputs: dict[str, Any] = field(default_factory=dict)

# =========================
# Quality Models
# =========================         
        
@dataclass
class QualityMetrics:
    roe: float | None = None
    roic: float | None = None
    gross_margin: float | None = None
    operating_margin: float | None = None
    net_margin: float | None = None
    debt_to_equity: float | None = None
    interest_coverage: float | None = None
    current_ratio: float | None = None
    quick_ratio: float | None = None
    fcf_margin: float | None = None
    revenue_cagr_1y: float | None = None    
    revenue_cagr_3y: float | None = None
    revenue_cagr_5y: float | None = None
    revenue_cagr_10y: float | None = None
    eps_cagr_1y: float | None = None
    eps_cagr_3y: float | None = None
    eps_cagr_5y: float | None = None
    eps_cagr_10y: float | None = None
    fcf_cagr_1y: float | None = None
    fcf_cagr_3y: float | None = None
    fcf_cagr_5y: float | None = None
    fcf_cagr_10y: float | None = None
    revenue_growth_3m: float | None = None
    revenue_growth_6m: float | None = None
    revenue_growth_9m: float | None = None
    eps_growth_3m: float | None = None
    eps_growth_6m: float | None = None
    eps_growth_9m: float | None = None
    ocf_growth_3m: float | None = None
    ocf_growth_6m: float | None = None
    ocf_growth_9m: float | None = None
    growth_status: dict[str, str] = field(default_factory=dict)
    growth_periods: dict[str, str] = field(default_factory=dict)
        
@dataclass(frozen=True)
class Recommendation:
    label: str
    stars: str
    score: float | None
        
# =========================
# Valuation Models
# =========================        
@dataclass
class ValuationMetrics:
    pe: float | None = None
    forward_pe: float | None = None          # market / Yahoo forward PE
    pb: float | None = None
    peg: float | None = None
    fcf_yield: float | None = None
    ev_ebit: float | None = None
    ev_ebitda: float | None = None
    dcf_value_per_share: float | None = None
    historical_pe: dict[str, float | None] | None = None
    historical_pb: dict[str, float | None] | None = None
    # ========= Historical PE/PB distribution =========
    historical_pe_stats: dict[str, float | None] | None = None
    historical_pb_stats: dict[str, float | None] | None = None
    historical_pe_percentile: float | None = None
    historical_pb_percentile: float | None = None    
    nowcast_pe_percentile: float | None = None   
    pe_percentile_5y: float | None = None
    pb_percentile_5y: float | None = None
    pe_sample_count: int = 0
    pe_history_years: float | None = None
        
@dataclass
class PriceTargets:

    # =========================
    # Model valuation
    # =========================
    model_buy_price: float | None
    model_fair_price: float | None
    model_sell_price: float | None

    # =========================
    # Historical PE valuation
    # =========================
    
    historical_P05_price: float | None = None
    historical_buy_price: float | None = None
    historical_fair_price: float | None = None
    historical_sell_price: float | None = None
    historical_P95_price: float | None = None    

    upside_to_fair: float | None = None

    primary_metric: str = ""
    basis: str = ""

# =========================
# Technical Analysis Models
# =========================

@dataclass
class TechnicalSignal:
    signal: str
    rsi: float | None
    short_ma: float | None
    long_ma: float | None
    support: float | None
    resistance: float | None

# =========================
# Market Flow / Chip Models
# =========================        
        
@dataclass
class USFlowProxySignal:
    signal: str
    volume_ratio: float | None
    short_ma: float | None
    long_ma: float | None
    return_20d: float | None
    reason: str



In [27]:
# 資料來源：Yahoo only
class YahooFinanceProvider:
    source_name = "Yahoo Finance (yfinance)"

    def __init__(self) -> None:
        self._ticker_cache: dict[str, Any] = {}
        self._analyst_consensus_cache: dict[str, dict[str, Any]] = {}

    def _ticker(self, ticker: str) -> Any:
        symbol = str(ticker).strip().upper()
        if symbol.endswith((".TW", ".TWO")) or symbol.isdigit():
            raise ValueError(f"US notebook does not accept Taiwan stock codes: {ticker}")
        if symbol not in self._ticker_cache:
            try:
                import yfinance as yf
            except ImportError as exc:
                raise ImportError("Please install yfinance: pip install yfinance") from exc
            self._ticker_cache[symbol] = yf.Ticker(symbol)
        return self._ticker_cache[symbol]

    @staticmethod
    def _value(frame: pd.DataFrame, label: str) -> float | None:
        if frame is None or frame.empty or label not in frame.index:
            return None
        values = frame.loc[label].dropna()
        return float(values.iloc[0]) if not values.empty else None

    @staticmethod
    def _row_values(frame: pd.DataFrame, label: str) -> list[float]:
        if frame is None or frame.empty or label not in frame.index:
            return []
        return [float(item) for item in frame.loc[label].dropna().tolist()]

    @staticmethod
    def _cell(frame: pd.DataFrame, label: str, column: object) -> float | None:
        if frame is None or frame.empty or label not in frame.index:
            return None
        value = frame.loc[label, column]
        return float(value) if pd.notna(value) else None

    @classmethod
    def _latest_ttm_eps(cls, quarterly_income: pd.DataFrame) -> float | None:
        if quarterly_income is None or quarterly_income.empty:
            return None

        quarterly_dates = sorted(quarterly_income.columns, reverse=True)
        values: list[float] = []
        for report_date in quarterly_dates:
            eps = cls._cell(quarterly_income, "Diluted EPS", report_date)
            if eps is None:
                eps = cls._cell(quarterly_income, "Basic EPS", report_date)
            if eps is None:
                net_income = cls._cell(quarterly_income, "Net Income", report_date)
                shares = cls._cell(quarterly_income, "Basic Average Shares", report_date)
                eps = safe_divide(net_income, shares)
            if eps is None:
                return None
            values.append(eps)
            if len(values) == 4:
                return float(sum(values))
        return None

    def _history(
        self,
        income: pd.DataFrame,
        cashflow: pd.DataFrame,
        quarterly_income: pd.DataFrame,
    ) -> dict[str, list[float]]:
        return {
            "revenue": self._row_values(income, "Total Revenue"),
            "net_income": self._row_values(income, "Net Income"),
            "eps": self._row_values(income, "Diluted EPS") or self._row_values(income, "Basic EPS"),
            "free_cash_flow": self._row_values(cashflow, "Free Cash Flow"),
            "quarterly_revenue": self._row_values(quarterly_income, "Total Revenue"),
            "quarterly_eps": self._row_values(quarterly_income, "Diluted EPS")
            or self._row_values(quarterly_income, "Basic EPS"),
        }

    def get_analyst_consensus(self, ticker: str) -> dict[str, Any]:
        """Yahoo current-month analyst counts and aggregate price targets; reference only."""
        from datetime import datetime, timezone

        symbol = str(ticker).strip().upper()
        if symbol in self._analyst_consensus_cache:
            return self._analyst_consensus_cache[symbol].copy()
        result = {
            "summary": "資料不足", "strong_buy": None, "buy": None,
            "hold": None, "sell": None, "strong_sell": None,
            "rating_count": None, "period": None,
            "target_low": None, "target_mean": None,
            "target_median": None, "target_high": None,
            "currency": None, "source": "Yahoo Finance (yfinance)",
            "symbol": symbol,
            "checked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "status": "資料不足",
        }
        errors = []
        try:
            instrument = self._ticker(symbol)
        except Exception as exc:
            result["status"] = f"Yahoo初始化失敗：{type(exc).__name__}"
            return result

        try:
            recommendations = instrument.get_recommendations()
            if isinstance(recommendations, pd.DataFrame) and not recommendations.empty:
                if "period" in recommendations.columns:
                    periods = recommendations["period"].astype(str)
                    current = recommendations.loc[periods.eq("0m")]
                    result["period"] = "0m" if not current.empty else str(recommendations.iloc[0].get("period"))
                else:
                    index_periods = pd.Index(recommendations.index).astype(str)
                    current = recommendations.loc[index_periods == "0m"]
                    result["period"] = "0m" if not current.empty else str(recommendations.index[0])
                if not current.empty:
                    row = current.iloc[0]
                    keys = ("strongBuy", "buy", "hold", "sell", "strongSell")
                    counts = []
                    for key in keys:
                        value = pd.to_numeric(pd.Series([row.get(key)]), errors="coerce").iloc[0]
                        counts.append(int(value) if pd.notna(value) and float(value) >= 0 else None)
                    if all(value is not None for value in counts):
                        (result["strong_buy"], result["buy"], result["hold"],
                         result["sell"], result["strong_sell"]) = counts
                        result["rating_count"] = sum(counts)
                        result["summary"] = "/".join(str(value) for value in counts)
                else:
                    errors.append("本月評等缺漏")
            else:
                errors.append("評等缺漏")
        except Exception as exc:
            errors.append(f"評等:{type(exc).__name__}")

        try:
            targets = instrument.get_analyst_price_targets()
            if isinstance(targets, dict):
                for result_key, yahoo_key in (("target_low", "low"), ("target_mean", "mean"),
                                              ("target_median", "median"), ("target_high", "high")):
                    value = pd.to_numeric(pd.Series([targets.get(yahoo_key)]), errors="coerce").iloc[0]
                    if pd.notna(value) and math.isfinite(float(value)) and float(value) > 0:
                        result[result_key] = float(value)
            if not any(result[key] is not None for key in ("target_low", "target_mean", "target_median", "target_high")):
                errors.append("目標價缺漏")
        except Exception as exc:
            errors.append(f"目標價:{type(exc).__name__}")

        try:
            info = instrument.get_info()
            if isinstance(info, dict):
                result["currency"] = info.get("currency") or info.get("financialCurrency")
        except Exception:
            pass
        ratings_ok = result["summary"] != "資料不足"
        targets_ok = any(result[key] is not None for key in ("target_low", "target_high"))
        result["status"] = "可用" if ratings_ok and targets_ok else "部分可用" if ratings_ok or targets_ok else "資料不足"
        if errors:
            result["status"] += "；" + "；".join(errors)
        self._analyst_consensus_cache[symbol] = result.copy()
        return result

    def get_snapshot(self, ticker: str) -> FinancialSnapshot:
        instrument = self._ticker(ticker)
        info = instrument.info
        income = instrument.financials
        balance = instrument.balance_sheet
        cashflow = instrument.cashflow
        quarterly_income = instrument.quarterly_income_stmt
        reference_notes = []
        try:
            quarterly_cashflow = instrument.quarterly_cashflow
        except Exception as exc:
            # Optional reference observations must not break the core valuation snapshot.
            quarterly_cashflow = pd.DataFrame()
            reference_notes.append(f"OCF quarterly request failed: {type(exc).__name__}")
        reference = self._quarterly_reference(quarterly_income, quarterly_cashflow)
        reference["notes"].extend(reference_notes)
        ttm_eps = self._latest_ttm_eps(quarterly_income)

        return FinancialSnapshot(
            ticker=ticker,
            source=self.source_name,
            as_of=date.today(),
            asset_type=info.get("quoteType"),
            current_price=info.get("currentPrice") or info.get("regularMarketPrice"),
            week_52_high=info.get("fiftyTwoWeekHigh"),
            week_52_low=info.get("fiftyTwoWeekLow"),
            market_cap=info.get("marketCap"),
            enterprise_value=info.get("enterpriseValue"),
            revenue=self._value(income, "Total Revenue"),
            gross_profit=self._value(income, "Gross Profit"),
            operating_income=self._value(income, "Operating Income"),
            net_income=self._value(income, "Net Income"),
            eps=ttm_eps if ttm_eps is not None else info.get("trailingEps"),
            ttm_eps=ttm_eps,
            forward_eps=info.get("forwardEps"),
            book_value=info.get("bookValue"),
            book_value_per_share=info.get("bookValue"),
            free_cash_flow=info.get("freeCashflow") or self._value(cashflow, "Free Cash Flow"),
            operating_cash_flow=self._value(cashflow, "Operating Cash Flow"),
            capex=self._value(cashflow, "Capital Expenditure"),
            total_debt=info.get("totalDebt") or self._value(balance, "Total Debt"),
            cash=info.get("totalCash"),
            shares_outstanding=info.get("sharesOutstanding"),
            total_equity=self._value(balance, "Stockholders Equity"),
            current_assets=self._value(balance, "Current Assets"),
            current_liabilities=self._value(balance, "Current Liabilities"),
            inventory=self._value(balance, "Inventory"),
            interest_expense=self._value(income, "Interest Expense"),
            financial_history=self._history(income, cashflow, quarterly_income),
            quarterly_reference=reference,
            cashflow_quality=self._cashflow_quality(reference, cashflow, info),
            fundamental_review_inputs=self._review_inputs(income, balance, info),
        )

    def get_price_history(self, ticker: str, period: str = "5y") -> pd.DataFrame:
        return self._ticker(ticker).history(period=period, auto_adjust=False)

    def get_historical_fundamentals(self, ticker: str) -> pd.DataFrame:
        """
        Return dated TTM EPS and BVPS observations.
        Priority:
        1. Quarterly statements -> TTM EPS + quarterly BVPS
        2. Fallback to annual statements if quarterly data is insufficient
        """
        instrument = self._ticker(ticker)
        q_income = getattr(instrument, "quarterly_income_stmt", None)
        q_balance = getattr(instrument, "quarterly_balance_sheet", None)

        rows: list[dict[str, float | pd.Timestamp | None]] = []

        # Quarterly path: build TTM EPS
        if q_income is not None and not q_income.empty:
            quarter_dates = list(q_income.columns)
            quarter_eps_map: dict[pd.Timestamp, float | None] = {}

            for col in quarter_dates:
                col_ts = pd.Timestamp(col)

                eps = self._cell(q_income, "Diluted EPS", col)
                if eps is None:
                    eps = self._cell(q_income, "Basic EPS", col)

                if eps is None:
                    net_income = self._cell(q_income, "Net Income", col)
                    shares = self._cell(q_income, "Basic Average Shares", col)
                    if net_income is not None and shares not in (None, 0):
                        eps = net_income / shares

                quarter_eps_map[col_ts] = eps

            sorted_dates = sorted(quarter_eps_map.keys())

            for i in range(3, len(sorted_dates)):
                report_date = sorted_dates[i]
                last_4 = sorted_dates[i - 3:i + 1]
                eps_values = [quarter_eps_map[d] for d in last_4]

                if any(v is None for v in eps_values):
                    continue

                ttm_eps = float(sum(eps_values))  # type: ignore[arg-type]

                bvps = None
                if q_balance is not None and not q_balance.empty and report_date in q_balance.columns:
                    equity = self._cell(q_balance, "Stockholders Equity", report_date)
                    shares = self._cell(q_balance, "Ordinary Shares Number", report_date)

                    if shares in (None, 0):
                        shares = self._cell(q_balance, "Share Issued", report_date)

                    if equity is not None and shares not in (None, 0):
                        bvps = equity / shares

                rows.append({
                    "report_date": report_date,
                    "ttm_eps": ttm_eps,
                    "bvps": bvps,
                })

        quarterly_dates = {pd.Timestamp(row["report_date"]) for row in rows}

        # Add annual history to extend the valuation lookback beyond Yahoo's limited quarterly history.
        income = getattr(instrument, "income_stmt", None)
        balance = getattr(instrument, "balance_sheet", None)

        if income is None or income.empty:
            return pd.DataFrame(rows).sort_values("report_date").reset_index(drop=True) if rows else pd.DataFrame(columns=["report_date", "ttm_eps", "bvps"])

        fallback_rows: list[dict[str, float | pd.Timestamp | None]] = []

        for col in income.columns:
            col_ts = pd.Timestamp(col)
            if col_ts in quarterly_dates:
                continue

            eps = self._cell(income, "Basic EPS", col)
            if eps is None:
                net_income = self._cell(income, "Net Income", col)
                shares = self._cell(income, "Basic Average Shares", col)
                if net_income is not None and shares not in (None, 0):
                    eps = net_income / shares

            bvps = None
            if balance is not None and not balance.empty and col in balance.columns:
                equity = self._cell(balance, "Stockholders Equity", col)
                shares = self._cell(balance, "Ordinary Shares Number", col)

                if shares in (None, 0):
                    shares = self._cell(balance, "Share Issued", col)

                if equity is not None and shares not in (None, 0):
                    bvps = equity / shares

            fallback_rows.append({
                "report_date": col_ts,
                "ttm_eps": eps,
                "bvps": bvps,
            })

        return pd.DataFrame(rows + fallback_rows).sort_values("report_date").reset_index(drop=True)



    

    @staticmethod
    def _quarterly_reference(income: pd.DataFrame, cashflow: pd.DataFrame) -> dict[str, Any]:
        """Yahoo quarterly EPS/revenue/OCF: single-quarter values, never YTD differences.

        Preserve gaps on a common fiscal-quarter timeline. Actual period-end dates,
        rather than calendar-quarter names, support non-December and 52/53-week years.
        Dates must be within 15 days of the latest endpoint minus 3*n months;
        ambiguous or missing endpoints remain unavailable, not interpolated.
        """
        notes = []

        def normalize(frame, label):
            if not isinstance(frame, pd.DataFrame) or frame.empty:
                notes.append(f"{label}: quarterly data unavailable")
                return pd.DataFrame()
            frame = frame.copy()
            dates = pd.to_datetime(frame.columns, errors="coerce", utc=True).tz_convert(None).normalize()
            frame.columns = dates
            frame = frame.loc[:, ~frame.columns.isna()]
            if frame.columns.duplicated().any():
                notes.append(f"{label}: duplicate period ends excluded")
                frame = frame.loc[:, ~frame.columns.duplicated(keep=False)]
            if frame.index.duplicated().any():
                notes.append(f"{label}: duplicate metric rows excluded")
                frame = frame.loc[~frame.index.duplicated(keep=False)]
            return frame

        income = normalize(income, "Income")
        cashflow = normalize(cashflow, "OCF")
        dates = sorted(set(income.columns) | set(cashflow.columns), reverse=True)
        result = {"quarters": [], "revenue": [], "eps": [], "ocf": [], "net_income": [], "capex": [], "notes": notes}
        if not dates:
            return result

        def number(frame, labels, endpoint):
            if endpoint is None or endpoint not in frame.columns:
                return None
            for label in labels:
                if label in frame.index:
                    value = pd.to_numeric(pd.Series([frame.at[label, endpoint]]), errors="coerce").iloc[0]
                    if pd.notna(value) and math.isfinite(value):
                        return float(value)
            return None

        for step in range(4):
            expected = dates[0] - pd.DateOffset(months=3 * step)
            candidates = [d for d in dates if abs((d - expected).days) <= 15]
            endpoint = candidates[0] if len(candidates) == 1 else None
            if endpoint is None:
                notes.append(f"Quarter offset {step}: missing or ambiguous period end")
            result["quarters"].append(endpoint.date().isoformat() if endpoint is not None else f"unavailable (~{expected.date().isoformat()})")
            result["revenue"].append(number(income, ["Total Revenue"], endpoint))
            result["eps"].append(number(income, ["Diluted EPS", "Basic EPS"], endpoint))
            result["ocf"].append(number(cashflow, ["Operating Cash Flow"], endpoint))
            result["net_income"].append(number(income, ["Net Income"], endpoint))
            result["capex"].append(number(cashflow, ["Capital Expenditure"], endpoint))
        for name in ("revenue", "eps", "ocf"):
            if any(value is None for value in result[name]):
                notes.append(f"{name}: incomplete recent quarterly observations")
        return result

    @staticmethod
    def _cashflow_quality(reference: dict[str, Any], annual_cashflow: pd.DataFrame, info: dict[str, Any]) -> dict[str, Any]:
        """Reference-only cash quality. FCF = OCF - nonnegative CapEx outflow.

        Yahoo Capital Expenditure is expected to be zero or negative. Positive
        values are ambiguous and are not silently converted with abs().
        Annual positivity is persistence, not a measure of cash-flow volatility.
        """
        notes = list(reference.get("notes", []))

        def finite(value):
            try:
                value = float(value)
                return value if math.isfinite(value) else None
            except (TypeError, ValueError):
                return None

        def capex_outflow(value):
            value = finite(value)
            return -value if value is not None and value <= 0 else None

        def total(values):
            clean = [finite(value) for value in values]
            return float(sum(clean)) if len(clean) == 4 and all(value is not None for value in clean) else None

        result = {
            "currency": info.get("financialCurrency"),
            "period": " / ".join(reversed(reference.get("quarters", []))),
            "ocf_ttm": total(reference.get("ocf", [])),
            "net_income_ttm": total(reference.get("net_income", [])),
            "revenue_ttm": total(reference.get("revenue", [])),
            "capex_ttm": total([capex_outflow(v) for v in reference.get("capex", [])]),
            "fcf_ttm": None, "ocf_to_net_income": None, "ocf_quality_status": "資料不足",
            "fcf_margin_ttm": None, "fcf_yield_ttm": None,
            "fcf_positive_ratio_5y": None, "fcf_positive_ratio_available": None,
            "fcf_positive_years": 0, "fcf_valid_years": 0,
            "fcf_positive_streak": None, "fcf_streak_status": "資料不足",
            "annual_fcf": [], "notes": notes,
        }
        if result["ocf_ttm"] is not None and result["capex_ttm"] is not None:
            result["fcf_ttm"] = result["ocf_ttm"] - result["capex_ttm"]
        profit = result["net_income_ttm"]
        if profit is not None and profit <= 0:
            result["ocf_quality_status"] = "淨利非正，不適用一般比率級距"
        elif profit is not None and result["ocf_ttm"] is not None:
            ratio = safe_divide(result["ocf_ttm"], profit)
            result["ocf_to_net_income"] = ratio
            if ratio is not None:
                result["ocf_quality_status"] = "高於1" if ratio > 1 else "0.8至1" if ratio >= .8 else "0.5至0.8以下" if ratio >= .5 else "低於0.5"
        revenue = result["revenue_ttm"]
        if revenue is not None and revenue > 0:
            result["fcf_margin_ttm"] = safe_divide(result["fcf_ttm"], revenue)
        market_cap = finite(info.get("marketCap"))
        if result["currency"] and result["currency"] == info.get("currency"):
            if market_cap is not None and market_cap > 0:
                result["fcf_yield_ttm"] = safe_divide(result["fcf_ttm"], market_cap)
        else:
            notes.append("財報與報價幣別不同或不明：TTM FCF殖利率留空，不自動換匯")
        for field in ("ocf_ttm", "net_income_ttm", "revenue_ttm", "capex_ttm"):
            if result[field] is None:
                notes.append(f"{field}: 最近四季資料不完整或數值不適用")
        if any(finite(v) is not None and finite(v) > 0 for v in reference.get("capex", [])):
            notes.append("季度 Capital Expenditure 為正值：支出口徑不明，未取絕對值")

        if not isinstance(annual_cashflow, pd.DataFrame) or annual_cashflow.empty:
            notes.append("年度現金流缺漏，無法評估FCF持續性")
            return result
        frame = annual_cashflow.copy()
        frame.columns = pd.to_datetime(frame.columns, errors="coerce", utc=True).tz_convert(None).normalize()
        frame = frame.loc[:, ~frame.columns.isna()]
        if frame.columns.duplicated().any():
            notes.append("年度現金流日期重複：排除歧義日期")
            frame = frame.loc[:, ~frame.columns.duplicated(keep=False)]
        if frame.index.duplicated().any():
            notes.append("年度現金流項目重複：排除歧義項目")
            frame = frame.loc[~frame.index.duplicated(keep=False)]
        dates = sorted(frame.columns, reverse=True)
        if not dates:
            return result
        for offset in range(5):
            expected = dates[0] - pd.DateOffset(years=offset)
            matches = [d for d in dates if abs((d - expected).days) <= 15]
            endpoint = matches[0] if len(matches) == 1 else None
            ocf = capex = None
            if endpoint is not None:
                if "Operating Cash Flow" in frame.index:
                    ocf = finite(frame.at["Operating Cash Flow", endpoint])
                if "Capital Expenditure" in frame.index:
                    raw = frame.at["Capital Expenditure", endpoint]
                    capex = capex_outflow(raw)
                    if finite(raw) is not None and finite(raw) > 0:
                        notes.append(f"年度 {endpoint.date()}: Capital Expenditure 為正值，FCF留空")
            fcf = ocf - capex if ocf is not None and capex is not None else None
            result["annual_fcf"].append({
                "period_end": endpoint.date().isoformat() if endpoint is not None else None,
                "expected_period_end": expected.date().isoformat(),
                "ocf": ocf, "capex": capex, "fcf": fcf,
            })
        values = [row["fcf"] for row in result["annual_fcf"]]
        valid = [v for v in values if v is not None]
        positive = sum(v > 0 for v in valid)
        result["fcf_positive_years"] = positive
        result["fcf_valid_years"] = len(valid)
        result["fcf_positive_ratio_available"] = safe_divide(positive, len(valid))
        if len(valid) == 5:
            result["fcf_positive_ratio_5y"] = positive / 5
        else:
            notes.append(f"五年度窗口僅 {len(valid)} 年有效，完整五年FCF正值比例留空")
        if values[0] is not None:
            streak = 0
            for value in values:
                if value is None or value <= 0:
                    break
                streak += 1
            result["fcf_positive_streak"] = streak
            result["fcf_streak_status"] = (
                "至少5年（僅檢查五年度窗口）" if streak == 5 else
                f"至少{streak}年；較早年度缺漏" if values[streak] is None else
                f"連續{streak}年；遇非正值年度停止"
            )
        return result

    @staticmethod
    def _review_inputs(income, balance, info):
        """Optional review inputs only: same fiscal-year income and balance sheet.

        Never use this dictionary to overwrite the original snapshot/scoring inputs.
        """
        result = {"sector": info.get("sector"), "industry": info.get("industry"),
                  "asset_type": info.get("quoteType"), "currency": info.get("financialCurrency"),
                  "period": None, "source": "Yahoo annual financials + balance_sheet", "note": ""}
        try:
            if income is None or balance is None or income.empty or balance.empty:
                result["note"] = "年度損益表或資產負債表缺漏"
                return result
            income_dates = {pd.Timestamp(c): c for c in income.columns}
            balance_dates = {pd.Timestamp(c): c for c in balance.columns}
            latest = max(income_dates)
            if latest != max(balance_dates):
                result["note"] = "最新年度損益表與資產負債表日期不一致，不混期計算"
                return result
            result["period"] = latest.date().isoformat()
            def cell(frame, column, names):
                for label in names:
                    if label not in frame.index:
                        continue
                    value = frame.loc[label, column]
                    if pd.notna(value) and not isinstance(value, bool):
                        value = float(value)
                        if math.isfinite(value): return value, label
                return None, None
            for key, names in {
                "debt": ["Total Debt"],
                "cash": ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments"],
                "current_assets": ["Current Assets"], "current_liabilities": ["Current Liabilities"],
            }.items():
                result[key], result[key+"_row"] = cell(balance, balance_dates[latest], names)
            for key, names in {"operating_income": ["Operating Income"],
                               "interest": ["Interest Expense"], "ebitda": ["EBITDA"]}.items():
                result[key], result[key+"_row"] = cell(income, income_dates[latest], names)
        except Exception as exc:
            result["period"] = None
            result["note"] = f"附加評價資料無法讀取：{type(exc).__name__}"
        return result


In [28]:
# 品質評估分數定義
# =========================
# Quality
# =========================

class QualityEngine:
    def calculate(self, snapshot: FinancialSnapshot) -> QualityMetrics:
        invested_capital = None
        if snapshot.total_equity is not None and snapshot.total_debt is not None and snapshot.cash is not None:
            invested_capital = snapshot.total_equity + snapshot.total_debt - snapshot.cash

        nopat = snapshot.operating_income * 0.8 if snapshot.operating_income is not None else None
        history = snapshot.financial_history

        return QualityMetrics(
            roe=safe_divide(snapshot.net_income, snapshot.total_equity),
            roic=safe_divide(nopat, invested_capital),
            gross_margin=safe_divide(snapshot.gross_profit, snapshot.revenue),
            operating_margin=safe_divide(snapshot.operating_income, snapshot.revenue),
            net_margin=safe_divide(snapshot.net_income, snapshot.revenue),
            debt_to_equity=safe_divide(snapshot.total_debt, snapshot.total_equity),
            interest_coverage=safe_divide(snapshot.operating_income, snapshot.interest_expense),
            current_ratio=safe_divide(snapshot.current_assets, snapshot.current_liabilities),
            quick_ratio=safe_divide(
                (snapshot.current_assets - snapshot.inventory)
                if snapshot.current_assets is not None and snapshot.inventory is not None
                else None,
                snapshot.current_liabilities,
            ),
            fcf_margin=safe_divide(snapshot.free_cash_flow, snapshot.revenue),
            revenue_cagr_1y=self._cagr(history.get("revenue", []), 1),
            revenue_cagr_3y=self._cagr(history.get("revenue", []), 3),
            revenue_cagr_5y=self._cagr(history.get("revenue", []), 5),
            revenue_cagr_10y=self._cagr(history.get("revenue", []), 10),
            eps_cagr_1y=self._cagr(history.get("eps", []), 1),
            eps_cagr_3y=self._cagr(history.get("eps", []), 3),
            eps_cagr_5y=self._cagr(history.get("eps", []), 5),
            eps_cagr_10y=self._cagr(history.get("eps", []), 10),
            fcf_cagr_1y=self._cagr(history.get("free_cash_flow", []), 1),
            fcf_cagr_3y=self._cagr(history.get("free_cash_flow", []), 3),
            fcf_cagr_5y=self._cagr(history.get("free_cash_flow", []), 5),
            fcf_cagr_10y=self._cagr(history.get("free_cash_flow", []), 10),
            **self._reference_metrics(snapshot),
        )

    @staticmethod
    def _cagr(values: list[float], years: int) -> float | None:
        if len(values) <= years or values[years] <= 0 or values[0] <= 0:
            return None
        return (values[0] / values[years]) ** (1 / years) - 1

    @staticmethod
    def _period_change(values: list[float | None], periods: int) -> tuple[float | None, str]:
        """1 => q0/q1-1; 2 => q1/q2-1; 3 => q2/q3-1. Reference only."""
        if periods < 1 or len(values) <= periods:
            return None, "資料不足"
        current, previous = values[periods - 1], values[periods]
        if current is None or previous is None or not math.isfinite(current) or not math.isfinite(previous):
            return None, "資料不足"
        if previous < 0:
            status = "由負轉正" if current > 0 else "由負轉零" if current == 0 else "負值改善" if current > previous else "負值惡化" if current < previous else "負值持平"
            return None, status
        if previous == 0:
            return None, "由零轉正" if current > 0 else "由零轉負" if current < 0 else "零值持平"
        rate = safe_divide(current, previous)
        if rate is None or not math.isfinite(rate - 1):
            return None, "比率無法計算"
        status = "由正轉負" if current < 0 else "成長" if current > previous else "衰退" if current < previous else "持平"
        return rate - 1, status

    @classmethod
    def _period_growth(cls, values: list[float | None], periods: int) -> float | None:
        return cls._period_change(values, periods)[0]

    @classmethod
    def _reference_metrics(cls, snapshot: FinancialSnapshot) -> dict[str, Any]:
        reference = snapshot.quarterly_reference
        result = {"growth_status": {}, "growth_periods": {}}
        for metric in ("revenue", "eps", "ocf"):
            values = reference.get(metric, [])
            for step, months in enumerate((3, 6, 9), start=1):
                name = f"{metric}_growth_{months}m"
                rate, status = cls._period_change(values, step)
                result[name] = rate
                result["growth_status"][name] = status
        quarters = reference.get("quarters", [])
        for step, months in enumerate((3, 6, 9), start=1):
            result["growth_periods"][f"{months}m"] = f"{quarters[step-1]} / {quarters[step]}" if len(quarters) > step else "資料不足"
        return result


# =========================
# Decision / scoring
# =========================


class DecisionEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.thresholds = config["scoring"]["recommendations"]

    def decide(self, score: float | None) -> Recommendation:
        if score is None:
            return Recommendation("Insufficient Data", "☆☆☆☆☆", None)
        if score >= self.thresholds["strong_buy"]:
            return Recommendation("Strong Buy", "★★★★★", score)
        if score >= self.thresholds["buy"]:
            return Recommendation("Buy", "★★★★☆", score)
        if score >= self.thresholds["hold"]:
            return Recommendation("Hold", "★★★☆☆", score)
        if score >= self.thresholds["reduce"]:
            return Recommendation("Reduce", "★★☆☆☆", score)
        return Recommendation("Avoid", "★☆☆☆☆", score)


class ScoringEngine:
    REFERENCE_ONLY = frozenset(
        f"{metric}_growth_{months}m"
        for metric in ("revenue", "eps", "ocf") for months in (3, 6, 9)
    )
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config
        
        
    
    

    def quality_score(self, metrics: Any, fcf_yield: float | None) -> float | None:
        data = asdict(metrics)
        data["fcf_yield"] = fcf_yield

        total = 0.0
        available_weight = 0.0

        for name, rule in self.config["quality_score"]["metrics"].items():
            if name in self.REFERENCE_ONLY:
                continue
            value = data.get(name)
            if value is None:
                continue

            weight = rule["weight"]
            excellent = rule["excellent"]
            good = rule["good"]
            lower = rule.get("lower_is_better", False)

            if lower:
                points = 1.0 if value <= excellent else 0.6 if value <= good else 0.2
            else:
                points = 1.0 if value >= excellent else 0.6 if value >= good else 0.2

            total += weight * points
            available_weight += weight

        return 100 * total / available_weight if available_weight else None
    
    
    
    





    @staticmethod
    def valuation_score(pe_percentile: float | None) -> float | None:
        return None if pe_percentile is None else max(0.0, min(100.0, 100 - pe_percentile))

    def growth_score(self, metrics: Any) -> float | None:
        rules = self.config.get("growth_score", {}).get("metrics", {})
        if not rules:
            return None

        total = 0.0
        available_weight = 0.0
        for name, rule in rules.items():
            if name in self.REFERENCE_ONLY:
                continue
            value = getattr(metrics, name, None)
            if value is None:
                continue

            weight = rule["weight"]
            excellent = rule["excellent"]
            good = rule["good"]
            lower = rule.get("lower_is_better", False)
            if lower:
                points = 1.0 if value <= excellent else 0.6 if value <= good else 0.2
            else:
                points = 1.0 if value >= excellent else 0.6 if value >= good else 0.2
            total += weight * points
            available_weight += weight

        return 100 * total / available_weight if available_weight else None
    
        #買賣時機分數  
    def technical_score(
        self,
        technical,
        price
    ):

        score = 0
        total_weight = 0


        # ==========================
        # 1. RSI (35%)
        # 越低越有買點價值
        # ==========================

        if technical.rsi is not None:

            total_weight += 35

            rsi = technical.rsi

            if rsi < 30:
                score += 35

            elif rsi < 40:
                score += 30

            elif rsi <=50:
                score += 25

            elif rsi <=60:
                score += 15

            elif rsi <=70:
                score += 5

            else:
                score += 0



        # ==========================
        # 2. 支撐距離 (35%)
        # 越靠近支撐越好
        # ==========================

        if technical.support is not None:

            total_weight +=35

            distance = (
                price - technical.support
            ) / price


            if distance <=0.02:
                score +=35

            elif distance <=0.05:
                score +=30

            elif distance <=0.10:
                score +=20

            else:
                score +=5



        # ==========================
        # 3. 壓力距離 (20%)
        # 避免追高
        # ==========================

        if technical.resistance is not None:

            total_weight +=20

            distance = (
                technical.resistance-price
            ) / price


            # 還有很大上漲空間
            if distance >=0.15:
                score +=20

            elif distance >=0.08:
                score +=15

            elif distance >=0.03:
                score +=8

            else:
                score +=0



        # ==========================
        # 4. 均線趨勢 (10%)
        # 趨勢只輔助
        # ==========================

        if (
            technical.short_ma is not None
            and technical.long_ma is not None
        ):

            total_weight +=10


            if technical.short_ma > technical.long_ma:
                score +=10

            elif technical.short_ma >= technical.long_ma*0.97:
                score +=5

            else:
                score +=0



        return (
            None
            if total_weight==0
            else score / total_weight *100
        )
    
    #籌碼面分數
    def flow_score(
        self,
        flow_signal: str | None,
        volume_ratio: float | None = None,
        return_20d: float | None = None,
    ) -> float | None:

        if flow_signal in {"Unavailable", "Insufficient Data"}:
            flow_signal = None


        score = 0
        total = 0


        # =====================
        # 資金訊號
        # =====================

        if flow_signal is not None:

            total += 40

            mapping = {

                "Strong Buy": 40,
                "Buy": 30,
                "Neutral": 20,
                "Sell": 10,
                "Sell/Reduce": 10,
                "Strong Sell": 0,

                # 中文防呆
                "強買":40,
                "買進":30,
                "中性":20,
                "賣出":10,
                "強賣":0,
            }

            score += mapping.get(
                flow_signal,
                20
            )


        # =====================
        # 成交量
        # =====================

        if volume_ratio is not None:

            total += 30


            if volume_ratio >= 1.5:
                score += 30

            elif volume_ratio >= 1.2:
                score += 20

            elif volume_ratio >= 1:
                score += 10

            else:
                score += 5



        # =====================
        # 20日報酬
        # =====================

        if return_20d is not None:

            total += 30


            if return_20d >= 0.10:
                score += 30

            elif return_20d >= 0.05:
                score += 20

            elif return_20d >= 0:
                score += 10

            else:
                score += 5



        # =====================
        # 防呆
        # =====================

        if total == 0:
            return None


        return score / total * 100
    

    def total_score(
        self,
        valuation: float | None,
        quality: float | None,
        growth: float | None,
        buffett: float | None,
    ) -> float | None:
        pairs = [
            ("valuation", valuation),
            ("quality", quality),
            ("growth", growth),
            ("buffett", buffett),
        ]
        weights = self.config["scoring"]["weights"]
        usable = [(weights[name], value) for name, value in pairs if value is not None]
        return None if not usable else sum(weight * value for weight, value in usable) / sum(weight for weight, _ in usable)    

class BuffettChecklist:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["buffett"]

    def evaluate(self, metrics: Any, snapshot: Any) -> dict[str, bool | None]:
        return {
            "roic": self._compare(metrics.roic, self.rules.get("roic_min"), True),
            "roe": self._compare(metrics.roe, self.rules.get("roe_min"), True),
            "debt_to_equity": self._compare(metrics.debt_to_equity, self.rules.get("debt_to_equity_max"), False),
            "fcf_positive": None if snapshot.free_cash_flow is None else snapshot.free_cash_flow > 0,
            "eps_growth_positive": None if metrics.eps_cagr_3y is None else metrics.eps_cagr_3y > 0,
            "operating_margin_positive": None if metrics.operating_margin is None else metrics.operating_margin > 0,
            "share_count_not_increasing": None,
        }

    @staticmethod
    def _compare(value: float | None, threshold: float | None, greater: bool) -> bool | None:
        if value is None or threshold is None:
            return None
        return value >= threshold if greater else value <= threshold

    @staticmethod
    def score(checks: dict[str, bool | None]) -> float | None:
        available = [value for value in checks.values() if value is not None]
        return None if not available else 100 * sum(available) / len(available)

class QuarterlyTrendReference:
    """Chronological text only; never produces a score."""
    @staticmethod
    def analyze(metrics: QualityMetrics) -> dict[str, str]:
        trends = {}
        for metric in ("eps", "revenue", "ocf"):
            states = [metrics.growth_status.get(f"{metric}_growth_{m}m", "資料不足") for m in (9, 6, 3)]
            trends[f"{metric}_direction"] = " → ".join(states)
        return trends


In [29]:
# 估值&定價系統
# =========================
# Valuation
# =========================
        
        
class ValuationEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config["valuation"]

    def calculate(
        self,
        snapshot: FinancialSnapshot,
        price_history: pd.DataFrame,
        historical_fundamentals: pd.DataFrame | None = None,
    ) -> ValuationMetrics:
        pe = safe_divide(snapshot.current_price, snapshot.eps)
        forward_pe = safe_divide(snapshot.current_price, snapshot.forward_eps) if snapshot.forward_eps is not None and snapshot.forward_eps > 0 else None  # Yahoo Forward EPS；非正值不計算本益比
        pb = safe_divide(snapshot.current_price, snapshot.book_value_per_share)
        growth = self._cagr(snapshot.financial_history.get("eps", []), 3)
        historical_ratios = self._point_in_time_ratios(price_history, historical_fundamentals) #歷史估值百分位 (年)
        historical_pe_stats = self._historical_ratio_statistics(historical_ratios, "pe", )  #歷史PE估值百分位 (季)
        historical_pb_stats = self._historical_ratio_statistics(historical_ratios, "pb", )  #歷史PB估值百分位 (季)
        
        pe_series = historical_ratios["pe"].dropna().tolist() if not historical_ratios.empty else []
        pb_series = historical_ratios["pb"].dropna().tolist() if not historical_ratios.empty else []
        nowcast_pe = snapshot.pe_ttm_nowcast
        nowcast_pe_percentile = percentile_rank(nowcast_pe, pe_series, )
        pe_dates = historical_ratios.loc[historical_ratios["pe"].notna(), "date"] if not historical_ratios.empty else pd.Series(dtype="datetime64[ns]")
        pe_history_years = (pe_dates.max() - pe_dates.min()).days / 365.25 if len(pe_dates) > 1 else None
        historical_pe = self._windowed_averages(historical_ratios, "pe")
        historical_pb = self._windowed_averages(historical_ratios, "pb")        
        # 新增：歷史PE序列
        historical_pe_values = (
            historical_ratios["pe"]
            .dropna()
            .tolist()
        )
        
        current_pe_percentile = None
        if (snapshot.current_price
            and snapshot.ttm_eps
            and snapshot.ttm_eps > 0
            and historical_pe_values
        ):
            current_pe = (
                snapshot.current_price /
                snapshot.ttm_eps
            )
            current_pe_percentile = (
                sum(
                    1 
                    for pe in historical_pe_values
                    if pe <= current_pe
                )
                /
                len(historical_pe_values)
            )

        return ValuationMetrics(
            pe=pe,
            forward_pe=forward_pe,
            pb=pb,
            peg=safe_divide(pe, growth) if growth and growth > 0 else None,
            fcf_yield=safe_divide(snapshot.free_cash_flow, snapshot.market_cap),
            ev_ebit=safe_divide(snapshot.enterprise_value, snapshot.operating_income),
            dcf_value_per_share=self._dcf(snapshot),
            historical_pe=historical_pe,
            historical_pb=historical_pb,
            historical_pe_stats=historical_pe_stats,
            historical_pb_stats=historical_pb_stats,
            historical_pe_percentile=current_pe_percentile,
            nowcast_pe_percentile=nowcast_pe_percentile,
            pe_percentile_5y=percentile_rank(pe, pe_series),
            pb_percentile_5y=percentile_rank(pb, pb_series),
            pe_sample_count=len(pe_series),
            pe_history_years=pe_history_years,
        )

    
    def _point_in_time_ratios(self, history: pd.DataFrame, fundamentals: pd.DataFrame | None) -> pd.DataFrame:
        if history is None or history.empty or fundamentals is None or fundamentals.empty or "Close" not in history:
            return pd.DataFrame(columns=["date", "pe", "pb"])

        prices = history[["Close"]].copy().sort_index()
        prices.index = pd.to_datetime(prices.index, utc=True).tz_convert(None)
        monthly_prices = prices["Close"].resample("ME").last().dropna().rename("close").reset_index()
        monthly_prices.columns = ["date", "close"]

        observations = fundamentals.copy()
        lag_days = self.config.get("earnings_availability_lag_days", 45)
        observations["effective_date"] = pd.to_datetime(observations["report_date"], utc=True).dt.tz_convert(None) + pd.DateOffset(days=lag_days)
        observations = observations.sort_values(["effective_date", "report_date"]).drop_duplicates("effective_date", keep="last")

        merged = pd.merge_asof(
            monthly_prices.sort_values("date"),
            observations[["effective_date", "ttm_eps", "bvps"]],
            left_on="date",
            right_on="effective_date",
            direction="backward",
        )
        merged["pe"] = [safe_divide(close, eps) if pd.notna(eps) and eps > 0 else None for close, eps in zip(merged["close"], merged["ttm_eps"])]
        merged["pb"] = [safe_divide(close, bvps) if pd.notna(bvps) and bvps > 0 else None for close, bvps in zip(merged["close"], merged["bvps"])]
        return merged[["date", "pe", "pb"]].dropna(how="all", subset=["pe", "pb"]).reset_index(drop=True)

    def _windowed_averages(self, ratios: pd.DataFrame, column: str) -> dict[str, float | None]:
        result: dict[str, float | None] = {}

        if ratios.empty:
            for year in self.config.get("historical_windows_years", []):
                result[f"{year}y"] = None
            for month in self.config.get("historical_window_months", []):
                result[f"{month}m"] = None
            return result

        latest = ratios["date"].max()

        for years in self.config.get("historical_windows_years", []):
            values = ratios.loc[ratios["date"] >= latest - pd.DateOffset(years=years), column].dropna()
            result[f"{years}y"] = float(values.mean()) if not values.empty else None

        for months in self.config.get("historical_window_months", []):
            values = ratios.loc[ratios["date"] >= latest - pd.DateOffset(months=months), column].dropna()
            result[f"{months}m"] = float(values.mean()) if not values.empty else None

        return result
        
    def _historical_ratio_statistics(
        self,
        ratios: pd.DataFrame,
        column: str,
    ) -> dict[str, float | None] | None:
        """
        對歷史 PE 或 PB 序列計算分位數統計。

        column:
          - "pe"
          - "pb"
        """
        if ratios is None or ratios.empty or column not in ratios.columns:
            return None

        values = ratios[column].dropna().astype(float)

        # 月頻資料至少 8 筆才建立價格帶
        if len(values) < 8:
            return None

        return {
            "mean": float(values.mean()),
            "p05": float(values.quantile(0.05)),
            "p25": float(values.quantile(0.25)),
            "p50": float(values.quantile(0.50)),
            "p75": float(values.quantile(0.75)),
            "p95": float(values.quantile(0.95)),
            "min": float(values.min()),
            "max": float(values.max()),
            "count": float(len(values)),
        }    
    
       
    def _dcf(self, snapshot: FinancialSnapshot) -> float | None:
        fcf = snapshot.free_cash_flow
        shares = snapshot.shares_outstanding
        if fcf is None or shares in (None, 0):
            return None

        assumptions = self.config["dcf"]
        rate = assumptions["discount_rate"]
        terminal = assumptions["terminal_growth_rate"]
        years = assumptions["forecast_years"]

        if rate <= terminal:
            return None

        projected = sum(fcf / ((1 + rate) ** year) for year in range(1, years + 1))
        terminal_value = fcf * (1 + terminal) / (rate - terminal) / ((1 + rate) ** years)

        return (projected + terminal_value) / shares

    @staticmethod
    def _cagr(values: list[float], years: int) -> float | None:
        if len(values) <= years or values[years] <= 0 or values[0] <= 0:
            return None
        return (values[0] / values[years]) ** (1 / years) - 1


# =========================
# Trading / profile
# =========================       

class ProfileResolver:
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config

    def classify(self, ticker: str, asset_type: str | None = None) -> Classification:
        override = self.config.get("ticker_classification", {}).get(ticker)
        if override:
            return Classification(market="US", sector=override["sector"])

        market = "US"
        sector = "traditional" if asset_type in {"ETF", "MUTUALFUND"} else "technology"
        return Classification(market=market, sector=sector)

    def rules(self, classification: Classification, ticker: str | None = None) -> dict[str, Any]:
        market = self.config["market_profiles"][classification.market]
        sector = self.config["sector_profiles"][classification.sector]
        override = self.config.get("ticker_rule_overrides", {}).get(ticker or "", {})
        return {
            "valuation": market["valuation"] | sector["valuation"] | override.get("valuation", {}),
            "trading": market["trading"] | override.get("trading", {}),
            "quality": sector["quality"] | override.get("quality", {}),
        }


class PriceTargetEngine:
    """
    依股票估值主指標產生價格帶：

    PE 型股票：
        歷史 PE 分位數 × TTM EPS

    PB 型股票：
        歷史 PB 分位數 × BVPS
    """

    @staticmethod
    def _normalized_base(
        snapshot: FinancialSnapshot,
        metric: str,
    ) -> float | None:
        """
        當沒有 Forward EPS Model 時，可作為 PE 型估值的備援基礎。
        PB 型股票直接使用最新 BVPS。
        """
        if metric == "pb":
            return snapshot.book_value_per_share

        # PE 型：優先使用歷史年度 EPS 中位數，
        # 沒有資料則使用目前 EPS。
        values = sorted(
            value
            for value in snapshot.financial_history.get("eps", [])
            if value is not None and value > 0
        )

        if not values:
            return snapshot.ttm_eps or snapshot.eps

        midpoint = len(values) // 2

        if len(values) % 2:
            return float(values[midpoint])

        return float(
            (values[midpoint - 1] + values[midpoint]) / 2
        )

    @staticmethod
    def _fair_multiple(
        valuation_metrics: ValuationMetrics,
        metric: str,
        cheap: float,
        expensive: float,
    ) -> float:
        """
        若有歷史平均估值，以歷史 3Y / 5Y / 1Y 平均為優先；
        否則使用 YAML cheap / expensive 的中間值。
        """
        historical = (
            valuation_metrics.historical_pe
            if metric == "pe"
            else valuation_metrics.historical_pb
        )

        if historical:
            for window in ("3y", "5y", "1y"):
                value = historical.get(window)

                if value is not None and value > 0:
                    return max(cheap, min(expensive, value))

        return (cheap + expensive) / 2

    @staticmethod
    def _get_historical_stats(
        valuation_metrics: ValuationMetrics,
        metric: str,
    ) -> dict[str, float | None] | None:
        """依估值主指標取得 PE 或 PB 的歷史分位數統計。"""
        if metric == "pe":
            return valuation_metrics.historical_pe_stats

        if metric == "pb":
            return valuation_metrics.historical_pb_stats

        return None

    @staticmethod
    def _get_trailing_base(
        snapshot: FinancialSnapshot,
        metric: str,
    ) -> float | None:
        """
        取得歷史價格帶的每股基礎：

        PE → TTM EPS，若無則 fallback EPS
        PB → 最新 BVPS
        """
        if metric == "pe":
            base = snapshot.ttm_eps or snapshot.eps
        elif metric == "pb":
            base = snapshot.book_value_per_share
        else:
            base = None

        if base is None or base <= 0:
            return None

        return float(base)

    def calculate(
        self,
        snapshot: FinancialSnapshot,
        rules: dict[str, Any],
        valuation_metrics: ValuationMetrics,
    ) -> PriceTargets:
        valuation = rules["valuation"]

        metric = valuation["primary_metric"].lower()
        cheap = valuation[f"{metric}_cheap"]
        expensive = valuation[f"{metric}_expensive"]

        # 依 metric 自動選 PE / PB 歷史統計
        historical_stats = self._get_historical_stats(
            valuation_metrics,
            metric,
        )

        # 歷史價格帶使用 TTM EPS 或 BVPS
        trailing_base = self._get_trailing_base(
            snapshot,
            metric,
        )

        # 模型價格帶的 PE 基礎：
        # 若另行提供即時 EPS，則優先使用；本版本預設為空，
        # 若無法計算，再退回正式 TTM EPS。
        if (
            metric == "pe"
            and snapshot.eps_ttm_nowcast is not None
            and snapshot.eps_ttm_nowcast > 0
        ):
            fair_base = snapshot.eps_ttm_nowcast
        else:
            fair_base = trailing_base

            
        # 若主要基礎資料缺失，再用中位數 EPS/BVPS 當 fallback
        if fair_base is None:
            fair_base = self._normalized_base(
                snapshot,
                metric,
            )

        # -------------------------------------------------
        # 決定模型估值倍數
        # 優先使用歷史 P25 / P50 / P75
        # -------------------------------------------------
        if (
            historical_stats is not None
            and historical_stats.get("p25") is not None
            and historical_stats.get("p50") is not None
            and historical_stats.get("p75") is not None
        ):
            cheap_multiple = historical_stats["p25"]
            fair_multiple = historical_stats["p50"]
            expensive_multiple = historical_stats["p75"]
        else:
            # 無足夠歷史資料時，退回 YAML 規則
            cheap_multiple = cheap
            fair_multiple = self._fair_multiple(
                valuation_metrics,
                metric,
                cheap,
                expensive,
            )
            expensive_multiple = expensive

        # -------------------------------------------------
        # 歷史分位數價格帶：P05 / P25 / P50 / P75 / P95
        # PE：TTM EPS × 歷史 PE 分位數
        # PB：BVPS × 歷史 PB 分位數
        # -------------------------------------------------
        historical_P05_price = None
        historical_buy_price = None
        historical_fair_price = None
        historical_sell_price = None
        historical_P95_price = None

        required_percentiles = ["p05", "p25", "p50", "p75", "p95"]

        if (
            historical_stats is not None
            and trailing_base is not None
            and all(
                historical_stats.get(key) is not None
                for key in required_percentiles
            )
        ):
            historical_P05_price = (
                trailing_base * historical_stats["p05"]
            )
            historical_buy_price = (
                trailing_base * historical_stats["p25"]
            )
            historical_fair_price = (
                trailing_base * historical_stats["p50"]
            )
            historical_sell_price = (
                trailing_base * historical_stats["p75"]
            )
            historical_P95_price = (
                trailing_base * historical_stats["p95"]
            )

        # -------------------------------------------------
        # 模型價格帶
        # -------------------------------------------------
        model_buy_price = (
            None
            if fair_base is None
            else fair_base * cheap_multiple
        )

        model_fair_price = (
            None
            if fair_base is None
            else fair_base * fair_multiple
        )

        model_sell_price = (
            None
            if fair_base is None
            else fair_base * expensive_multiple
        )

        upside_to_fair = (
            safe_divide(
                model_fair_price - snapshot.current_price,
                snapshot.current_price,
            )
            if (
                model_fair_price is not None
                and snapshot.current_price not in (None, 0)
            )
            else None
        )

        # 說明文字依 PE/PB 改變
        if metric == "pe":
            historical_basis = "historical PE percentile × TTM EPS"
            model_basis = (
                "model uses forward EPS when available"
            )
        elif metric == "pb":
            historical_basis = "historical PB percentile × BVPS"
            model_basis = "model uses latest BVPS"
        else:
            historical_basis = "historical valuation percentile"
            model_basis = "model valuation"

        return PriceTargets(
            model_buy_price=model_buy_price,
            model_fair_price=model_fair_price,
            model_sell_price=model_sell_price,

            historical_P05_price=historical_P05_price,
            historical_buy_price=historical_buy_price,
            historical_fair_price=historical_fair_price,
            historical_sell_price=historical_sell_price,
            historical_P95_price=historical_P95_price,

            upside_to_fair=upside_to_fair,
            primary_metric=metric.upper(),
            basis=(
                f"{metric.upper()} valuation; "
                f"{model_basis}; "
                f"historical uses {historical_basis}"
            ),
        )
    

In [30]:
# 舊技術／資金流引擎定義暫存；基本面批次流程不會呼叫。
#技術分析&籌碼 金流分析
# =========================
# TechnicalAnalysis & USFlowProxy 
# =========================
class TechnicalAnalysisEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["technical"]

    def analyze(self, prices: pd.DataFrame) -> TechnicalSignal:
        if prices is None or prices.empty or "Close" not in prices:
            return TechnicalSignal("Unavailable", None, None, None, None, None)

        close = prices["Close"].dropna()
        minimum = max(self.rules["long_ma_days"], self.rules["rsi_days"] + 1)

        if len(close) < minimum:
            return TechnicalSignal("Insufficient Data", None, None, None, None, None)

        short_ma = float(close.tail(self.rules["short_ma_days"]).mean())
        long_ma = float(close.tail(self.rules["long_ma_days"]).mean())

        delta = close.diff()
        gains = delta.clip(lower=0)
        losses = -delta.clip(upper=0)

        avg_gain = gains.ewm(alpha=1 / self.rules["rsi_days"], adjust=False).mean().iloc[-1]
        avg_loss = losses.ewm(alpha=1 / self.rules["rsi_days"], adjust=False).mean().iloc[-1]

        rsi = 100.0 if avg_loss == 0 else float(100 - 100 / (1 + avg_gain / avg_loss))
        support = float(close.tail(self.rules["support_lookback_days"]).min())
        resistance = float(close.tail(self.rules["resistance_lookback_days"]).max())
        latest = float(close.iloc[-1])

        if short_ma > long_ma and rsi <= self.rules["rsi_oversold"]:
            signal = "Buy Setup"
        elif short_ma < long_ma or rsi >= self.rules["rsi_overbought"]:
            signal = "Sell/Reduce Setup"
        elif latest <= support * 1.02:
            signal = "Watch Buy Zone"
        else:
            signal = "Hold/Wait"

        return TechnicalSignal(signal, rsi, short_ma, long_ma, support, resistance)
    
class USFlowProxyEngine:
    """
    US large-money / institutional-flow proxy using price and volume.
    """
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config.get("us_flow_proxy", {
            "short_ma_days": 20,
            "long_ma_days": 60,
            "volume_short_days": 5,
            "volume_long_days": 60,
            "volume_ratio_threshold": 1.3,
            "return_lookback_days": 20,
        })

    def analyze(self, prices: pd.DataFrame) -> USFlowProxySignal:
        if prices is None or prices.empty or "Close" not in prices or "Volume" not in prices:
            return USFlowProxySignal(
                signal="Unavailable",
                volume_ratio=None,
                short_ma=None,
                long_ma=None,
                return_20d=None,
                reason="Price/volume history unavailable",
            )

        close = prices["Close"].dropna()
        volume = prices["Volume"].dropna()

        minimum = max(
            self.rules["long_ma_days"],
            self.rules["volume_long_days"],
            self.rules["return_lookback_days"] + 1,
        )

        if len(close) < minimum or len(volume) < minimum:
            return USFlowProxySignal(
                signal="Insufficient Data",
                volume_ratio=None,
                short_ma=None,
                long_ma=None,
                return_20d=None,
                reason="Not enough price/volume history",
            )

        short_ma = float(close.tail(self.rules["short_ma_days"]).mean())
        long_ma = float(close.tail(self.rules["long_ma_days"]).mean())
        latest = float(close.iloc[-1])

        short_vol = float(volume.tail(self.rules["volume_short_days"]).mean())
        long_vol = float(volume.tail(self.rules["volume_long_days"]).mean())
        volume_ratio = safe_divide(short_vol, long_vol)

        lookback = self.rules["return_lookback_days"]
        past_price = float(close.iloc[-lookback - 1])
        return_20d = safe_divide(latest - past_price, past_price)

        bullish = (
            latest > short_ma > long_ma and
            (volume_ratio is not None and volume_ratio >= self.rules["volume_ratio_threshold"]) and
            (return_20d is not None and return_20d > 0)
        )

        bearish = (
            latest < short_ma < long_ma and
            (volume_ratio is not None and volume_ratio >= self.rules["volume_ratio_threshold"]) and
            (return_20d is not None and return_20d < 0)
        )

        vol_text = f"{volume_ratio:.2f}" if volume_ratio is not None else "NA"
        ret_text = f"{return_20d:.2%}" if return_20d is not None else "NA"

        if bullish:
            return USFlowProxySignal(
                signal="Buy",
                volume_ratio=volume_ratio,
                short_ma=short_ma,
                long_ma=long_ma,
                return_20d=return_20d,
                reason=f"Uptrend with expanding volume (vol_ratio={vol_text}, return_20d={ret_text})",
            )

        if bearish:
            return USFlowProxySignal(
                signal="Sell/Reduce",
                volume_ratio=volume_ratio,
                short_ma=short_ma,
                long_ma=long_ma,
                return_20d=return_20d,
                reason=f"Downtrend with expanding volume (vol_ratio={vol_text}, return_20d={ret_text})",
            )

        return USFlowProxySignal(
            signal="Hold",
            volume_ratio=volume_ratio,
            short_ma=short_ma,
            long_ma=long_ma,
            return_20d=return_20d,
            reason=f"Mixed/neutral flow proxy (vol_ratio={vol_text}, return_20d={ret_text})",
        )




In [31]:
# 報表產出&中英對照
# =========================
# Report / translations
# =========================
ZH_TW_COLUMNS = {
    "Ticker": "代號",
    "Source": "資料來源",
    "Market": "市場",
    "Sector": "產業",
    "Price": "現價",
    "PE": "本益比",
    "TTM EPS": "近年EPS",
    "Forward EPS": "預估EPS",   
    "Forward PE": "預估本益比",    
    "PB": "股價淨值比",
    "ROIC": "投入資本報酬率",
    "ROE": "股東權益報酬率",
    "FCF Yield": "既有估值FCF殖利率", #FCF per Share/Price 
    "Revenue Growth 3M": "營收近三月成長率",
    "Revenue Growth 6M": "營收近六月成長率",
    "EPS Growth 3M": "EPS 近三月成長率",
    "EPS Growth 6M": "EPS 近六月成長率",
    "Revenue Growth 3M Status": "營收 3M狀態",
    "Revenue Growth 6M Status": "營收 6M狀態",
    "Revenue Growth 9M": "營收 近九月成長率",
    "Revenue Growth 9M Status": "營收 9M狀態",
    "EPS Growth 3M Status": "EPS 3M狀態",
    "EPS Growth 6M Status": "EPS 6M狀態",
    "EPS Growth 9M": "EPS 近九月成長率",
    "EPS Growth 9M Status": "EPS 9M狀態",
    "OCF Growth 3M": "OCF 近三月成長率",
    "OCF Growth 3M Status": "OCF 3M狀態",
    "OCF Growth 6M": "OCF 近六月成長率",
    "OCF Growth 6M Status": "OCF 6M狀態",
    "OCF Growth 9M": "OCF 近九月成長率",
    "OCF Growth 9M Status": "OCF 9M狀態",
    "Growth 3M Period": "3M比較季度",
    "Growth 6M Period": "6M比較季度",
    "Growth 9M Period": "9M比較季度",
    "OCF Momentum Trend": "OCF逐季趨勢",
    "Reference Notes": "季增率資料註記",
    "CF Financial Currency": "現金流財報幣別",
    "CF TTM Period": "現金流TTM四季期間",
    "OCF TTM": "營業現金流OCF_TTM",
    "Net Income TTM CF Basis": "同期淨利_TTM",
    "Revenue TTM CF Basis": "同期營收_TTM",
    "CapEx TTM Outflow": "資本支出_TTM_正值支出",
    "FCF TTM": "自由現金流FCF_TTM",
    "OCF / Net Income TTM": "盈餘現金轉換率_OCF除以淨利",
    "OCF Quality Status": "盈餘現金轉換狀態",
    "FCF Margin TTM": "FCF利潤率_TTM",
    "FCF Yield TTM Aligned": "FCF殖利率_TTM同幣別",
    "FCF Positive Ratio 5Y": "FCF正值比例_完整五年度",
    "FCF Positive Ratio Available": "FCF正值比例_可用年度",
    "FCF Positive Years": "FCF正值年度數",
    "FCF Valid Years": "FCF有效年度數_最多五年",
    "FCF Positive Streak": "FCF可確認連續正值年數",
    "FCF Streak Status": "FCF連續正值說明",
    "Cash Flow Quality Notes": "現金流品質資料註記",
    "Annual FCF 1 Period": "FCF年度1期末",
    "Annual FCF 1": "FCF年度1金額",
    "Annual FCF 2 Period": "FCF年度2期末",
    "Annual FCF 2": "FCF年度2金額",
    "Annual FCF 3 Period": "FCF年度3期末",
    "Annual FCF 3": "FCF年度3金額",
    "Annual FCF 4 Period": "FCF年度4期末",
    "Annual FCF 4": "FCF年度4金額",
    "Annual FCF 5 Period": "FCF年度5期末",
    "Annual FCF 5": "FCF年度5金額",
    "EPS CAGR 3Y": "EPS 三年複合成長率",
    "EPS Momentum Trend": "EPS短期趨勢",
    "Revenue Momentum Trend": "營收短期趨勢",
    "Valuation Percentile": "歷史估值百分位",
    "Valuation Sample Count": "歷史估值樣本數",
    "Valuation History Years": "歷史估值可用年數",
    "Growth Score": "成長分數",
    "Quality Score": "品質分數",
    "Buffett Score": "巴菲特檢核分數",
    "Fundamental Score": "基本面分數",
    "Recommendation": "基本面投資建議",
    "Price Recommendation": "買價投資建議",
    "Earnings Recommendation": "盈餘投資建議",
    "Price Attractiveness Score": "買價吸引力分數",
    "Combined Valuation Percentile": "綜合估值百分位",
    "Price Recommendation Note": "買價建議依據",
    "Earnings Trend Score": "盈餘趨勢分數",
    "Earnings Recommendation Note": "盈餘建議依據",
    "Cash Quality Review": "現金獲利品質評價",
    "Financial Safety Review": "財務安全性評價",
    "Analyst Consensus": "分析師共識（強買/買進/持有/賣出/強賣）",
    "Analyst Strong Buy Count": "分析師評等_強買家數",
    "Analyst Buy Count": "分析師評等_買進家數",
    "Analyst Hold Count": "分析師評等_持有家數",
    "Analyst Sell Count": "分析師評等_賣出家數",
    "Analyst Strong Sell Count": "分析師評等_強賣家數",
    "Analyst Rating Count": "分析師評等_樣本總數",
    "Analyst Rating Period": "分析師評等_期間",
    "Analyst Target Low": "分析師目標價最低",
    "Analyst Target Mean": "分析師目標價平均",
    "Analyst Target Median": "分析師目標價中位數",
    "Analyst Target High": "分析師目標價最高",
    "Analyst Target Low Upside": "分析師最低目標價上行空間",
    "Analyst Target Mean Upside": "分析師平均目標價上行空間",
    "Analyst Target Median Upside": "分析師中位目標價上行空間",
    "Analyst Target High Upside": "分析師最高目標價上行空間",
    "Analyst Target Currency": "分析師目標價幣別",
    "Analyst Data Source": "分析師資料來源",
    "Analyst Retrieved UTC": "分析師資料取得時間_UTC",
    "Analyst Yahoo Symbol": "分析師資料_Yahoo代號",
    "Analyst Data Status": "分析師資料狀態",
    "Cash Quality Review Note": "現金獲利品質評價依據",
    "Financial Safety Review Note": "財務安全性評價依據",
    "Review Sector": "附加評價_Yahoo產業",
    "Review Industry": "附加評價_Yahoo子產業",
    "Safety Period": "財務安全_年度財報期末",
    "Safety Source": "財務安全_資料來源",
    "Safety Currency": "財務安全_財報幣別",
    "Safety Debt": "財務安全_年度有息負債",
    "Safety Cash": "財務安全_年度現金",
    "Safety Cash Row": "財務安全_現金採用項目",
    "Safety Operating Income": "財務安全_年度營業利益",
    "Safety Interest": "財務安全_年度利息支出",
    "Safety EBITDA": "財務安全_年度EBITDA",
    "Safety Current Assets": "財務安全_年度流動資產",
    "Safety Current Liabilities": "財務安全_年度流動負債",
    "Safety Net Debt": "財務安全_淨負債",
    "Safety Net Debt EBITDA": "財務安全_淨負債除以EBITDA",
    "Safety Interest Coverage": "財務安全_營業利益除以利息",
    "Safety Current Ratio": "財務安全_流動比率",
    "Review Rule Version": "附加評價_規則版本",
    "Stars": "星等",
    "Model Buy Price": "模型預估便宜價",
    "Model Fair Price": "模型預估合理價",
    "Model Sell Price": "模型預估昂貴價",
    "Historical P05 Price": "歷史估值05%價格",
    "Historical Buy Price": "歷史估值25%價格",
    "Historical Fair Price": "歷史估值50%價格",
    "Historical Sell Price": "歷史估值75%價格",
    "Historical P95 Price": "歷史估值95%價格",
    "Fair Value Upside": "合理價上行空間",
    "Target Metric": "目標價依據",
    "Valuation Basis": "估值模型說明",
    "Technical Signal": "技術面訊號",
    "Technical Score": "技術面分數",
    "RSI": "相對強弱指標 RSI",
    "Support": "支撐價",
    "Resistance": "壓力價",
    "Flow Signal": "資金流訊號",
    "Flow Score": "資金流分數",
    "Flow Note": "資金流說明",
    "Error": "錯誤",
}

ZH_TW_VALUES = {
    "technology": "科技",
    "financial": "金融",
    "traditional": "傳產",
    "cyclical": "景氣循環",
    "Strong Buy": "強力買進",
    "Buy": "買進",
    "Hold": "持有",
    "Reduce": "減碼",
    "Avoid": "避開",
    "Insufficient Data": "資料不足",
    "Data Error": "資料錯誤",
    "Buy Setup": "買進訊號形成",
    "Sell/Reduce Setup": "賣出／減碼訊號",
    "Watch Buy Zone": "觀察買點區",
    "Hold/Wait": "持有／等待",
    "Unavailable": "暫無資料",
    "Sell/Reduce": "賣出／減碼",
    "PE": "本益比",
    "PB": "股價淨值比",
    "Provider has no institutional-flow data": "資料來源未提供法人／籌碼資料",
    "Not enough recent institutional-flow observations": "近期籌碼資料不足",
    "Recent institutional net-buying trend": "近期法人買賣超趨勢",
    "Price/volume history unavailable": "無價格或成交量資料",
    "Not enough price/volume history": "價格或成交量歷史不足",
    "Uptrend with expanding volume": "放量上升趨勢",
    "Downtrend with expanding volume": "放量下跌趨勢",
    "Mixed/neutral flow proxy": "資金流訊號中性",
    "N/A": "無法計算",
}

#格式輸出調整
def format_report(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 先將所有數值欄位保留到小數點後 3 位
    numeric_cols = df.select_dtypes(include=["number"]).columns
    df[numeric_cols] = df[numeric_cols].round(3)

    # 再把評分欄位改成整數
    score_cols = [
        "Valuation Percentile",
        "Growth Score",
        "Quality Score",
        "Buffett Score",
        "Fundamental Score",
        "RSI",
    ]

    for col in score_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").round(0).astype("Int64")

    return df

class IndependentInvestmentAdvice:
    """獨立參考評價，不回寫基本面分數、星等、定價或其他建議。"""
    PRICE_WEIGHTS = (0.60, 0.40)
    TREND_WEIGHTS = {"revenue": 0.25, "eps": 0.35, "ocf": 0.40}
    PERIOD_WEIGHTS = {3: 0.50, 6: 0.30, 9: 0.20}

    @staticmethod
    def _number(value):
        if value is None or isinstance(value, bool):
            return None
        try:
            value = float(value)
            return value if math.isfinite(value) else None
        except (TypeError, ValueError):
            return None

    @classmethod
    def price(cls, historical, nowcast):
        values = [cls._number(historical), cls._number(nowcast)]
        values = [v if v is not None and 0 <= v <= 100 else None for v in values]
        valid = [(v, w) for v, w in zip(values, cls.PRICE_WEIGHTS) if v is not None]
        if not valid:
            return {"score": None, "percentile": None, "advice": "資料不足", "note": "兩個估值百分位皆無有效資料"}
        percentile = sum(v*w for v,w in valid) / sum(w for _,w in valid)
        score = 100 - percentile
        if all(v is not None for v in values) and abs(values[0] - values[1]) > 30:
            return {"score": score, "percentile": percentile, "advice": "估值分歧", "note": "兩估值百分位差距超過30點，暫不給買賣建議"}
        advice = "低點強買" if percentile < 15 else "買進" if percentile < 50 else "持有" if percentile < 85 else "高點賣出"
        note = "歷史60%＋即時40%" if len(valid) == 2 else "僅依歷史估值百分位" if values[0] is not None else "僅依即時歷史估值位階"
        return {"score": score, "percentile": percentile, "advice": advice, "note": note}

    @staticmethod
    def earnings_label(score):
        return "持續改善" if score >= 80 else "偏向改善" if score >= 60 else "表現分歧" if score >= 40 else "偏向衰退" if score >= 20 else "持續衰退"

    @classmethod
    def earnings(cls, quality, reference):
        # 不以缺漏或負基期當成0分，也不讓剩餘資料自動升權重。
        points = {"成長": 100, "持平": 50, "衰退": 0}
        component_scores, unavailable, special = {}, [], []
        for metric, weight in cls.TREND_WEIGHTS.items():
            component = 0.0
            for months, period_weight in cls.PERIOD_WEIGHTS.items():
                key = f"{metric}_growth_{months}m"
                state = quality.growth_status.get(key, "資料不足")
                rate = cls._number(getattr(quality, key, None))
                if state in {"資料不足", "比率無法計算"}:
                    unavailable.append(key)
                elif state not in points:
                    special.append(f"{key}={state}")
                elif rate is None:
                    unavailable.append(key)
                else:
                    component += points[state] * period_weight
            component_scores[metric] = component
        for metric in ("eps", "ocf"):
            values = reference.get(metric, [])
            latest = cls._number(values[0]) if len(values) else None
            if latest is None:
                unavailable.append(f"{metric}:最新季缺值")
            elif latest <= 0:
                special.append(f"{metric}:最新季非正值")
        if unavailable:
            return {"score": None, "advice": "資料不足", "note": "；".join(unavailable + special)}
        if special:
            return {"score": None, "advice": "盈餘待觀察", "note": "；".join(special)}
        score = sum(component_scores[metric] * weight for metric, weight in cls.TREND_WEIGHTS.items())
        return {"score": score, "advice": cls.earnings_label(score), "note": "營收25%／EPS35%／OCF40%；該季50%／上季30%／上兩季20%。僅評趨勢，非盈餘品質鑑證"}

class FundamentalRiskReviews:
    """初始篩選規則；與基本面評分、買價及盈餘趨勢建議分離。"""
    RULES = {
        "cash_good_conversion": 1.0, "cash_weak_conversion": 0.5,
        "min_fcf_years": 3, "good_fcf_positive_ratio": 0.8,
        "safe_interest_coverage": 5.0, "risk_interest_coverage": 1.0,
        "safe_net_debt_ebitda": 2.0, "risk_net_debt_ebitda": 4.0,
        "safe_current_ratio": 1.0, "max_annual_age_days": 550,
    }
    _number = staticmethod(IndependentInvestmentAdvice._number)

    @classmethod
    def applicability(cls, inputs):
        sector = str(inputs.get("sector") or "").lower()
        industry = str(inputs.get("industry") or "").lower()
        asset = str(inputs.get("asset_type") or "").upper()
        if asset and asset != "EQUITY": return "不適用", "ETF／基金等非一般企業不套用此規則"
        if "financial" in sector or "reit" in industry or any(s in industry for s in ("bank", "insurance", "capital markets", "credit services")):
            return "不適用", "金融業或REIT須另用產業專屬口徑"
        if not sector or not asset: return "資料不足", "Yahoo產業或資產類型缺漏，無法確認適用性"
        return None, "一般非金融企業初始檢核，非產業相對評分"

    @classmethod
    def cash_quality(cls, cf, inputs):
        status, scope = cls.applicability(inputs)
        if status: return {"label": status, "note": scope}
        try:
            latest = date.fromisoformat(cf["period"].split(" / ")[-1])
            age = (date.today() - latest).days
            if age < 0 or age > 200:
                return {"label":"資料不足","note":"最新現金流季度日期異常或距今超過200天"}
        except (KeyError, TypeError, ValueError, AttributeError):
            return {"label":"資料不足","note":"缺少有效TTM現金流期間"}
        get=lambda key:cls._number(cf.get(key))
        ni,ocf,ratio,margin,years,positive = [get(k) for k in (
            "net_income_ttm","ocf_ttm","ocf_to_net_income","fcf_margin_ttm","fcf_valid_years","fcf_positive_ratio_available")]
        if ni is not None and ni <= 0:
            return {"label":"待觀察","note":"同期淨利非正，不用OCF／淨利評為良好；虧損不等同會計品質有問題"}
        if ocf is not None and ocf <= 0:
            return {"label":"偏弱","note":"TTM營業現金流非正；依已知風險提示，其他項目可能缺漏"}
        if ratio is not None and ratio < cls.RULES["cash_weak_conversion"]:
            return {"label":"偏弱","note":"OCF／同期淨利低於0.5；依已知風險提示，其他項目可能缺漏"}
        if any(v is None for v in (ni,ocf,ratio,margin,years,positive)):
            return {"label":"資料不足","note":"需完整TTM現金流／淨利／營收及年度FCF，不將缺值當零"}
        if years < cls.RULES["min_fcf_years"]:
            return {"label":"資料不足","note":f"可用FCF年度僅{int(years)}年；至少需3年（最多觀察最近5年度）"}
        try:
            annual_age = (date.today() - date.fromisoformat(cf["annual_fcf"][0]["period_end"])).days
            if annual_age < 0 or annual_age > cls.RULES["max_annual_age_days"]:
                return {"label":"資料不足","note":"年度FCF歷史過舊或日期異常"}
        except (KeyError, IndexError, TypeError, ValueError):
            return {"label":"資料不足","note":"年度FCF日期缺漏"}
        if not 0 <= positive <= 1:
            return {"label":"資料不足","note":"FCF正值比例不在有效範圍"}
        findings=[]
        if ratio < cls.RULES["cash_good_conversion"]: findings.append("OCF／同期淨利低於1")
        if margin <= 0: findings.append("FCF Margin非正，需檢視投資支出，並不直接代表盈餘造假")
        if positive < cls.RULES["good_fcf_positive_ratio"]: findings.append("可用年度FCF正值比例低於80%")
        return {"label":"注意" if findings else "良好", "note":"；".join(findings) if findings else f"OCF／淨利≥1、FCF Margin>0、{int(years)}個可用年度FCF正值比例≥80%；僅現金轉換與持續性檢核"}

    @classmethod
    def safety(cls, inputs):
        result={"label":"資料不足","note":"", "net_debt":None,"net_debt_ebitda":None,
                "interest_coverage":None,"current_ratio":None}
        status,scope=cls.applicability(inputs)
        if status:
            return {**result,"label":status,"note":scope}
        try:
            age=(date.today()-date.fromisoformat(inputs["period"])).days
        except (TypeError,ValueError,KeyError):
            return {**result,"note":inputs.get("note") or "缺少可對齊的年度財報日期"}
        if age < 0 or age > cls.RULES["max_annual_age_days"]:
            return {**result,"note":"年度財報日期異常或距今超過550天，不評為穩健"}
        fields=("debt","cash","operating_income","interest","ebitda","current_assets","current_liabilities")
        vals={key:cls._number(inputs.get(key)) for key in fields}
        for key in ("debt","cash","interest","current_assets","current_liabilities"):
            if vals[key] is not None and vals[key]<0: vals[key]=None
        debt,cash,op,interest,ebitda,assets,liab=[vals[k] for k in fields]
        if debt is not None and cash is not None:
            result["net_debt"]=debt-cash
            if ebitda is not None and ebitda>0:result["net_debt_ebitda"]=(debt-cash)/ebitda
        if op is not None and interest is not None and interest>0:result["interest_coverage"]=op/interest
        if assets is not None and liab is not None and liab>0:result["current_ratio"]=assets/liab
        risks=[]
        if result["interest_coverage"] is not None and result["interest_coverage"]<cls.RULES["risk_interest_coverage"]:risks.append("營業利益不足支付利息（低於高風險門檻）")
        if result["net_debt_ebitda"] is not None and result["net_debt_ebitda"]>cls.RULES["risk_net_debt_ebitda"]:risks.append("淨負債／EBITDA超過高風險門檻")
        if result["net_debt"] is not None and result["net_debt"]>0 and ebitda is not None and ebitda<=0:risks.append("淨負債為正且EBITDA非正")
        missing=[key for key,v in vals.items() if v is None]
        if risks:return {**result,"label":"高風險","note":"；".join(risks)+("；另缺漏："+",".join(missing) if missing else "")}
        if missing:return {**result,"note":"年度財報缺漏／數值無效："+",".join(missing)}
        cautions=[]
        if ebitda<=0:cautions.append("EBITDA非正，不以負分母判定低槓桿")
        if interest==0:
            if debt>0:cautions.append("有負債但利息為零，無法由保障倍數確認偿債能力")
            elif op<=0:cautions.append("雖無負債但營業利益非正")
        elif result["interest_coverage"]<cls.RULES["safe_interest_coverage"]:cautions.append("利息保障低於穩健門檻")
        if result["net_debt_ebitda"] is not None and result["net_debt_ebitda"]>cls.RULES["safe_net_debt_ebitda"]:cautions.append("淨負債／EBITDA高於穩健門檻")
        if result["current_ratio"] is not None and result["current_ratio"]<cls.RULES["safe_current_ratio"]:cautions.append("流動比率低於穩健門檻")
        return {**result,"label":"注意" if cautions else "穩健", "note":"；".join(cautions) if cautions else "同年度檢核：利息保障≥5倍（或無債且無利息）、淨負債／EBITDA≤2、流動比率≥1（或無流動負債）；不代表未來無風險"}


def order_advice_columns(report):
    """兩種報表均固定為：星等、基本面投資建議、買價投資建議、盈餘投資建議。"""
    advice = ["Recommendation", "Price Recommendation", "Earnings Recommendation", "Cash Quality Review", "Financial Safety Review", "Analyst Consensus", "Analyst Target Low", "Analyst Target High"]
    columns = [c for c in report.columns if c not in advice]
    if "Stars" not in columns:
        columns.append("Stars")
    index = columns.index("Stars") + 1
    columns[index:index] = advice
    return report.reindex(columns=columns)


def analyze_universe(
    provider: YahooFinanceProvider,
    tickers: list[str],
    config: dict[str, Any],
    language: str = "en",
) -> pd.DataFrame:
    valuation_engine = ValuationEngine(config)
    quality_engine = QualityEngine()
    score_engine = ScoringEngine(config)
    checklist = BuffettChecklist(config)
    decision_engine = DecisionEngine(config)
    profiles = ProfileResolver(config)
    targets = PriceTargetEngine()

    records: list[dict[str, Any]] = []

    for ticker in tickers:
        try:
            snapshot = provider.get_snapshot(ticker)
            prices = provider.get_price_history(ticker)
            historical_fundamentals = provider.get_historical_fundamentals(ticker)

            valuation = valuation_engine.calculate(snapshot, prices, historical_fundamentals)
            quality = quality_engine.calculate(snapshot)

            quality_score = score_engine.quality_score(quality, valuation.fcf_yield)
            valuation_score = score_engine.valuation_score(valuation.pe_percentile_5y)
            growth_score = score_engine.growth_score(quality)
            short_growth_direction = QuarterlyTrendReference.analyze(quality)

            checks = checklist.evaluate(quality, snapshot)
            buffett_score = checklist.score(checks)

            total_score = score_engine.total_score(
                valuation_score,
                quality_score,
                growth_score,
                buffett_score,
            )

            classification = profiles.classify(ticker, snapshot.asset_type)
            rules = profiles.rules(classification, ticker)

            target = targets.calculate(snapshot, rules, valuation)
            decision = decision_engine.decide(total_score)
            price_advice = IndependentInvestmentAdvice.price(valuation.pe_percentile_5y, None)
            earnings_advice = IndependentInvestmentAdvice.earnings(quality, snapshot.quarterly_reference)
            review_inputs = snapshot.fundamental_review_inputs
            cash_review = FundamentalRiskReviews.cash_quality(snapshot.cashflow_quality, review_inputs)
            safety_review = FundamentalRiskReviews.safety(review_inputs)
            analyst = provider.get_analyst_consensus(ticker)

                
            annual_cf = (snapshot.cashflow_quality.get("annual_fcf", []) + [{}] * 5)[:5]
            records.append({
                "Ticker": ticker,
                "Source": snapshot.source,
                "Market": classification.market,
                "Sector": classification.sector,
                "Price": snapshot.current_price,                
                "PE": valuation.pe,
                # 已公告最近四季 EPS
                "TTM EPS": snapshot.ttm_eps if snapshot.ttm_eps is not None else "N/A",
                # Yahoo / 市場共識預估
                "Forward EPS": snapshot.forward_eps if snapshot.forward_eps is not None else "N/A",
                "Forward PE": valuation.forward_pe if valuation.forward_pe is not None else "N/A",
                "PB": valuation.pb,
                "ROIC": quality.roic,
                "ROE": quality.roe,
                "FCF Yield": valuation.fcf_yield,
                "Revenue Growth 3M": quality.revenue_growth_3m,
                "Revenue Growth 6M": quality.revenue_growth_6m,
                "EPS Growth 3M": quality.eps_growth_3m,
                "EPS Growth 6M": quality.eps_growth_6m,
                "Revenue Growth 3M Status": quality.growth_status["revenue_growth_3m"],
                "Revenue Growth 6M Status": quality.growth_status["revenue_growth_6m"],
                "Revenue Growth 9M": quality.revenue_growth_9m,
                "Revenue Growth 9M Status": quality.growth_status["revenue_growth_9m"],
                "EPS Growth 3M Status": quality.growth_status["eps_growth_3m"],
                "EPS Growth 6M Status": quality.growth_status["eps_growth_6m"],
                "EPS Growth 9M": quality.eps_growth_9m,
                "EPS Growth 9M Status": quality.growth_status["eps_growth_9m"],
                "OCF Growth 3M": quality.ocf_growth_3m,
                "OCF Growth 3M Status": quality.growth_status["ocf_growth_3m"],
                "OCF Growth 6M": quality.ocf_growth_6m,
                "OCF Growth 6M Status": quality.growth_status["ocf_growth_6m"],
                "OCF Growth 9M": quality.ocf_growth_9m,
                "OCF Growth 9M Status": quality.growth_status["ocf_growth_9m"],
                "Growth 3M Period": quality.growth_periods["3m"],
                "Growth 6M Period": quality.growth_periods["6m"],
                "Growth 9M Period": quality.growth_periods["9m"],
                "OCF Momentum Trend": short_growth_direction["ocf_direction"],
                "Reference Notes": "; ".join(snapshot.quarterly_reference.get("notes", [])),
                "CF Financial Currency": snapshot.cashflow_quality.get("currency"),
                "CF TTM Period": snapshot.cashflow_quality.get("period"),
                "OCF TTM": snapshot.cashflow_quality.get("ocf_ttm"),
                "Net Income TTM CF Basis": snapshot.cashflow_quality.get("net_income_ttm"),
                "Revenue TTM CF Basis": snapshot.cashflow_quality.get("revenue_ttm"),
                "CapEx TTM Outflow": snapshot.cashflow_quality.get("capex_ttm"),
                "FCF TTM": snapshot.cashflow_quality.get("fcf_ttm"),
                "OCF / Net Income TTM": snapshot.cashflow_quality.get("ocf_to_net_income"),
                "OCF Quality Status": snapshot.cashflow_quality.get("ocf_quality_status"),
                "FCF Margin TTM": snapshot.cashflow_quality.get("fcf_margin_ttm"),
                "FCF Yield TTM Aligned": snapshot.cashflow_quality.get("fcf_yield_ttm"),
                "FCF Positive Ratio 5Y": snapshot.cashflow_quality.get("fcf_positive_ratio_5y"),
                "FCF Positive Ratio Available": snapshot.cashflow_quality.get("fcf_positive_ratio_available"),
                "FCF Positive Years": snapshot.cashflow_quality.get("fcf_positive_years"),
                "FCF Valid Years": snapshot.cashflow_quality.get("fcf_valid_years"),
                "FCF Positive Streak": snapshot.cashflow_quality.get("fcf_positive_streak"),
                "FCF Streak Status": snapshot.cashflow_quality.get("fcf_streak_status"),
                "Cash Flow Quality Notes": "; ".join(snapshot.cashflow_quality.get("notes", [])),
                "Annual FCF 1 Period": annual_cf[0].get("period_end"),
                "Annual FCF 1": annual_cf[0].get("fcf"),
                "Annual FCF 2 Period": annual_cf[1].get("period_end"),
                "Annual FCF 2": annual_cf[1].get("fcf"),
                "Annual FCF 3 Period": annual_cf[2].get("period_end"),
                "Annual FCF 3": annual_cf[2].get("fcf"),
                "Annual FCF 4 Period": annual_cf[3].get("period_end"),
                "Annual FCF 4": annual_cf[3].get("fcf"),
                "Annual FCF 5 Period": annual_cf[4].get("period_end"),
                "Annual FCF 5": annual_cf[4].get("fcf"),
                "EPS CAGR 3Y": quality.eps_cagr_3y,
                "EPS Momentum Trend": short_growth_direction["eps_direction"], #額外評價
                "Revenue Momentum Trend": short_growth_direction["revenue_direction"], #額外評價
                "Valuation Percentile": valuation.pe_percentile_5y,
                #"Valuation Sample Count": valuation.pe_sample_count,
                #"Valuation History Years": valuation.pe_history_years,
                "Growth Score": growth_score,
                "Quality Score": quality_score,
                "Buffett Score": buffett_score,
                "Fundamental Score": total_score,
                "Recommendation": decision.label,
                "Stars": decision.stars,
                "Price Recommendation": price_advice["advice"],
                "Earnings Recommendation": earnings_advice["advice"],
                "Cash Quality Review": cash_review["label"],
                "Financial Safety Review": safety_review["label"],
                "Analyst Consensus": analyst["summary"],
                "Analyst Strong Buy Count": analyst["strong_buy"],
                "Analyst Buy Count": analyst["buy"],
                "Analyst Hold Count": analyst["hold"],
                "Analyst Sell Count": analyst["sell"],
                "Analyst Strong Sell Count": analyst["strong_sell"],
                "Analyst Rating Count": analyst["rating_count"],
                "Analyst Rating Period": analyst["period"],
                "Analyst Target Low": analyst["target_low"],
                "Analyst Target Mean": analyst["target_mean"],
                "Analyst Target Median": analyst["target_median"],
                "Analyst Target High": analyst["target_high"],
                "Analyst Target Low Upside": safe_divide(analyst["target_low"], snapshot.current_price) - 1 if safe_divide(analyst["target_low"], snapshot.current_price) is not None else None,
                "Analyst Target Mean Upside": safe_divide(analyst["target_mean"], snapshot.current_price) - 1 if safe_divide(analyst["target_mean"], snapshot.current_price) is not None else None,
                "Analyst Target Median Upside": safe_divide(analyst["target_median"], snapshot.current_price) - 1 if safe_divide(analyst["target_median"], snapshot.current_price) is not None else None,
                "Analyst Target High Upside": safe_divide(analyst["target_high"], snapshot.current_price) - 1 if safe_divide(analyst["target_high"], snapshot.current_price) is not None else None,
                "Analyst Target Currency": analyst["currency"],
                "Analyst Data Source": analyst["source"],
                "Analyst Retrieved UTC": analyst["checked_at"],
                "Analyst Yahoo Symbol": analyst["symbol"],
                "Analyst Data Status": analyst["status"],
                "Model Buy Price": target.model_buy_price,
                "Model Fair Price": target.model_fair_price,
                "Model Sell Price": target.model_sell_price,
                "Historical P05 Price": target.historical_P05_price,
                "Historical Buy Price": target.historical_buy_price,
                "Historical Fair Price": target.historical_fair_price,
                "Historical Sell Price": target.historical_sell_price,
                "Historical P95 Price": target.historical_P95_price,
                "Fair Value Upside": target.upside_to_fair,
                "Target Metric": target.primary_metric,
                "Price Attractiveness Score": price_advice["score"],
                "Combined Valuation Percentile": price_advice["percentile"],
                "Price Recommendation Note": price_advice["note"],
                "Earnings Trend Score": earnings_advice["score"],
                "Earnings Recommendation Note": earnings_advice["note"],
                "Cash Quality Review Note": cash_review["note"],
                "Financial Safety Review Note": safety_review["note"],
                "Review Sector": review_inputs.get("sector"),
                "Review Industry": review_inputs.get("industry"),
                "Safety Period": review_inputs.get("period"),
                "Safety Source": review_inputs.get("source"),
                "Safety Currency": review_inputs.get("currency"),
                "Safety Debt": review_inputs.get("debt"),
                "Safety Cash": review_inputs.get("cash"),
                "Safety Cash Row": review_inputs.get("cash_row"),
                "Safety Operating Income": review_inputs.get("operating_income"),
                "Safety Interest": review_inputs.get("interest"),
                "Safety EBITDA": review_inputs.get("ebitda"),
                "Safety Current Assets": review_inputs.get("current_assets"),
                "Safety Current Liabilities": review_inputs.get("current_liabilities"),
                "Safety Net Debt": safety_review["net_debt"],
                "Safety Net Debt EBITDA": safety_review["net_debt_ebitda"],
                "Safety Interest Coverage": safety_review["interest_coverage"],
                "Safety Current Ratio": safety_review["current_ratio"],
                "Review Rule Version": "US-general-v1",
                #"Valuation Basis": target.basis,
            })
        except Exception as exc:
            LOGGER.exception("Could not analyze %s", ticker)
            records.append({
                "Ticker": ticker,
                "Source": None,
                "Recommendation": "Data Error",
                "Price Recommendation": "資料不足",
                "Earnings Recommendation": "資料不足",
                "Cash Quality Review": "資料不足",
                "Financial Safety Review": "資料不足",
                "Analyst Consensus": "資料不足",
                "Analyst Data Status": "股票分析失敗",
                "Error": str(exc),
            })

    report = pd.DataFrame(records)
    report = format_report(order_advice_columns(report))

    if language.lower() in {"zh", "zh-tw", "zh_tw"}:
        return report.replace(ZH_TW_VALUES).rename(columns=ZH_TW_COLUMNS)

    return report


CASHFLOW_REPORT_KEYS = ['Ticker', 'Source', 'CF Financial Currency', 'CF TTM Period', 'OCF TTM', 'Net Income TTM CF Basis', 'Revenue TTM CF Basis', 'CapEx TTM Outflow', 'FCF TTM', 'OCF / Net Income TTM', 'OCF Quality Status', 'FCF Margin TTM', 'FCF Yield TTM Aligned', 'FCF Positive Ratio 5Y', 'FCF Positive Ratio Available', 'FCF Positive Years', 'FCF Valid Years', 'FCF Positive Streak', 'FCF Streak Status', 'Cash Flow Quality Notes', 'Annual FCF 1 Period', 'Annual FCF 1', 'Annual FCF 2 Period', 'Annual FCF 2', 'Annual FCF 3 Period', 'Annual FCF 3', 'Annual FCF 4 Period', 'Annual FCF 4', 'Annual FCF 5 Period', 'Annual FCF 5', 'OCF Growth 3M', 'OCF Growth 6M', 'OCF Growth 9M', 'OCF Momentum Trend']

# 簡化版欄位：依使用者範例固定名稱、順序；不得插入說明欄。
US_SIMPLE_COLUMNS = [
    ("Ticker", "代號"),
    ("Sector", "產業"),
    ("Price", "現價"),
    ("Valuation Percentile", "歷史估值百分位"),
    ("Growth Score", "成長分數"),
    ("Quality Score", "品質分數"),
    ("Buffett Score", "巴菲特檢核分數"),
    ("Fundamental Score", "基本面分數"),
    ("Stars", "基本面評價星等"),
    ("Recommendation", "基本面投資評價"),
    ("Price Recommendation", "買價投資評價"),
    ("Earnings Recommendation", "盈餘趨勢評價"),
    ("Cash Quality Review", "現金獲利品質評價"),
    ("Financial Safety Review", "財務安全性評價"),
    ("Analyst Consensus", "分析師共識（強買/買進/持有/賣出/強賣）"),
    ("Analyst Target Low", "分析師目標價最低"),
    ("Analyst Target High", "分析師目標價最高"),
    ("Model Buy Price", "模型預估便宜價"),
    ("Model Fair Price", "模型預估合理價"),
    ("Model Sell Price", "模型預估昂貴價"),
    ("Historical P05 Price", "歷史估值05%價格"),
    ("Historical Buy Price", "歷史估值25%價格"),
    ("Historical Fair Price", "歷史估值50%價格"),
    ("Historical Sell Price", "歷史估值75%價格"),
    ("Historical P95 Price", "歷史估值95%價格"),
    ("Fair Value Upside", "合理價上行空間"),
    ("Target Metric", "目標價依據"),
    ("Revenue Momentum Trend", "營收逐季趨勢"),
    ("EPS Momentum Trend", "EPS逐季趨勢"),
    ("OCF Momentum Trend", "OCF逐季趨勢"),
    ("PE", "本益比"),
    ("Forward PE", "預估本益比"),
    ("TTM EPS", "近年EPS"),
    ("Forward EPS", "預估EPS"),
    ("PB", "股價淨值比"),
    ("ROIC", "投入資本報酬率"),
    ("ROE", "股東權益報酬率"),
    ("FCF Yield", "自由現金流殖利率"),
    ("Revenue Growth 3M", "該季營收成長率"),
    ("Revenue Growth 6M", "上一季營收成長率"),
    ("Revenue Growth 9M", "上兩季營收成長率"),
    ("EPS Growth 3M", "該季EPS成長率"),
    ("EPS Growth 6M", "上季EPS成長率"),
    ("EPS Growth 9M", "上兩季EPS成長率"),
    ("OCF Growth 3M", "該季OCF成長率"),
    ("OCF Growth 6M", "上季OCF成長率"),
    ("OCF Growth 9M", "上兩季OCF成長率"),
]


def build_simple_report(report: pd.DataFrame) -> pd.DataFrame:
    """同一份分析結果選欄、改名；不重算、不排序、不變更數值或百分比格式。"""
    source = report.copy()
    if "代號" in source.columns:
        source = source.rename(columns={value: key for key, value in ZH_TW_COLUMNS.items()})
    elif "Ticker" in source.columns:
        source = source.replace(ZH_TW_VALUES)
    elif not source.empty:
        raise ValueError("Report must contain Ticker or 代號")
    # 全部股票失敗或清單為空時，仍輸出固定表頭；失敗原因保留在詳細版。
    simple = source.reindex(columns=[key for key, _ in US_SIMPLE_COLUMNS]).copy()
    simple.columns = [label for _, label in US_SIMPLE_COLUMNS]
    return simple


def save_report(report, config, prefix=None):
    """一次分析、兩份 Excel；詳細版與簡化版使用完全獨立的目錄。"""
    from datetime import datetime
    from pathlib import Path
    from openpyxl.styles import Alignment
    from openpyxl.utils import get_column_letter

    config_name = str(config.get("_config_name", "US"))
    file_prefix = str(prefix or config_name)
    # config 名稱不可變成路徑，確保兩種版本不會因設定而寫到其他位置。
    for name in (config_name, file_prefix):
        if not name or name in {".", ".."} or any(c in name for c in '/\\:<>"|?*'):
            raise ValueError("Report name must be a filename, not a path")
    now = datetime.now()
    root = Path("reports")
    stamp = now.strftime("%Y%m%d_%H%M%S_%f")
    tables = {"詳細版": report.copy(), "簡化版": build_simple_report(report)}
    paths = {}
    for version, table in tables.items():
        output_dir = root / version / config_name / now.strftime("%Y") / now.strftime("%m")
        output_dir.mkdir(parents=True, exist_ok=True)
        filename = output_dir / f"{file_prefix}_Report_{stamp}.xlsx"
        with pd.ExcelWriter(filename, engine="openpyxl") as writer:
            table.to_excel(writer, sheet_name="Report", index=False, freeze_panes=(1, 2))
            if version == "詳細版":
                rules = pd.DataFrame([
                    {"規則": key, "設定值": value} for key, value in FundamentalRiskReviews.RULES.items()
                ])
                notes = pd.DataFrame([
                    {"規則": "現金獲利品質", "設定值": "淨利非正待觀察；OCF非正或OCF/淨利<0.5偏弱；資料完整且OCF/淨利≥1、FCF Margin>0、至少3年度且FCF正值比例≥80%為良好，其餘注意"},
                    {"規則": "財務安全性", "設定值": "利息保障<1、淨負債/EBITDA>4、或淨負債正且EBITDA非正為高風險；完整資料達5倍/2倍/流動比率1門檻為穩健，其餘注意"},
                    {"規則": "資料與適用性", "設定值": "只適用一般非金融企業；金融業/REIT/基金不適用；缺值不補零，已知高風險優先提示。安全檢核使用同年度財報，最新TTM季度最多200天、年度最多550天"},
                    {"規則": "分析師共識與目標價", "設定值": "Yahoo Finance當月（0m）五分類家數：強買/買進/持有/賣出/強賣；目標價保留最低、平均、中位數與最高。簡化版只顯示五分類共識及最低/最高目標價"},
                    {"規則": "分析師資料限制", "設定值": "屬券商分析師彙總共識，不代表台灣投信買賣超；可能缺漏、延遲或受極端目標價影響，只作獨立參考，不納入WHID分數與投資建議"},
                    {"規則": "限制", "設定值": "初始人工門檻未經回測，不是信用評等或會計品質鑑證；未考慮債務到期分布、授信額度與產業相對門檻；不影響原分數與投資建議"},
                    {"規則": "指標來源（門檻由本模型設定）", "設定值": "https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2026/financial-analysis-techniques"},
                    {"規則": "現金品質分析參考", "設定值": "https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2026/evaluating-quality-financial-reports"},
                ])
                pd.concat([notes, rules], ignore_index=True).to_excel(writer, sheet_name="ReviewRules", index=False)
                columns = []
                for key in CASHFLOW_REPORT_KEYS:
                    for candidate in (key, ZH_TW_COLUMNS.get(key, key)):
                        if candidate in table.columns and candidate not in columns:
                            columns.append(candidate)
                if len(columns) > 2:
                    table[columns].to_excel(writer, sheet_name="CashFlowQuality", index=False, freeze_panes=(1, 2))
            for sheet in writer.sheets.values():
                sheet.auto_filter.ref = sheet.dimensions
                sheet.row_dimensions[1].height = 42
                for cell in sheet[1]:
                    label = str(cell.value)
                    width = 29 if "趨勢" in label or "Trend" in label else 18
                    if "註記" in label or "狀態" in label or "Note" in label or "Status" in label:
                        width = 25
                    sheet.column_dimensions[cell.column_letter].width = width
                    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                if sheet.title == "ReviewRules":
                    sheet.column_dimensions["A"].width = 38
                    sheet.column_dimensions["B"].width = 100
                    for row in sheet.iter_rows(min_row=2):
                        for cell in row:
                            cell.alignment = Alignment(vertical="top", wrap_text=True)
                        sheet.row_dimensions[row[0].row].height = 60
        paths[version] = filename
        print(f"{version}已儲存：{filename.resolve()}")
    return paths


In [32]:
#------------------------------------------------------------------
#Cell 1 — 初始化
#------------------------------------------------------------------
from pathlib import Path
import pandas as pd

# 顯示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

# 載入設定與 provider / engines
config = load_config("config/US.yaml")
provider = YahooFinanceProvider()

valuation_engine = ValuationEngine(config)
quality_engine = QualityEngine()
score_engine = ScoringEngine(config)
checklist = BuffettChecklist(config)
decision_engine = DecisionEngine(config)
profiles = ProfileResolver(config)
targets = PriceTargetEngine()

print("Initialized.")

Initialized.


In [33]:
# 美股基本面與估值：一次分析，分開輸出詳細版與簡化版。
tickers = config["universe"]["us"]
report = analyze_universe(provider, tickers, config, language="zh-tw")
report_paths = save_report(report, config)
simple_report = build_simple_report(report)
simple_report


詳細版已儲存：C:\Users\User\Project\WHID\Stock_Valuation\reports\詳細版\US\2026\09\US_Report_20260901_195412_327328.xlsx
簡化版已儲存：C:\Users\User\Project\WHID\Stock_Valuation\reports\簡化版\US\2026\09\US_Report_20260901_195412_327328.xlsx


,代號,產業,現價,歷史估值百分位,成長分數,品質分數,巴菲特檢核分數,基本面分數,基本面評價星等,基本面投資評價,買價投資評價,盈餘趨勢評價,現金獲利品質評價,財務安全性評價,模型預估便宜價,模型預估合理價,模型預估昂貴價,歷史估值05%價格,歷史估值25%價格,歷史估值50%價格,歷史估值75%價格,歷史估值95%價格,合理價上行空間,目標價依據,營收逐季趨勢,EPS逐季趨勢,OCF逐季趨勢,本益比,預估本益比,近年EPS,預估EPS,股價淨值比,投入資本報酬率,股東權益報酬率,自由現金流殖利率,該季營收成長率,上一季營收成長率,上兩季營收成長率,該季EPS成長率,上季EPS成長率,上兩季EPS成長率,該季OCF成長率,上季OCF成長率,上兩季OCF成長率
0,GOOG,科技,335.41,0,83,90,100,90,★★★★★,強力買進,低點強買,表現分歧,注意,穩健,497.818,568.338,596.112,391.925,497.818,568.338,596.112,761.728,0.694,本益比,成長 → 衰退 → 成長,衰退 → 成長 → 成長,成長 → 衰退 → 衰退,16.846,22.621,19.910,14.828,6.590,0.352,0.318,0.006,0.090,-0.035,0.112,0.783,0.812,-0.017,-0.147,-0.126,0.082
1,META,科技,572.34,7,79,85,100,86,★★★★★,強力買進,低點強買,表現分歧,良好,穩健,698.780,796.932,917.383,546.504,698.780,796.932,917.383,1082.394,0.392,本益比,成長 → 衰退 → 成長,成長 → 成長 → 衰退,成長 → 衰退 → 衰退,21.557,16.371,26.550,34.960,5.583,0.278,0.278,0.015,0.080,-0.060,0.169,-0.408,0.176,7.457,-0.011,-0.110,0.207
2,MSFT,科技,507.29,11,75,90,100,86,★★★★★,強力買進,低點強買,持續改善,良好,穩健,583.423,638.292,700.126,484.485,583.423,638.292,700.126,783.483,0.258,本益比,成長 → 成長 → 成長,成長 → 衰退 → 成長,衰退 → 成長 → 成長,28.246,21.520,17.960,23.573,8.517,0.251,0.302,0.004,0.086,0.020,0.046,0.126,-0.172,0.387,0.188,0.305,-0.206
3,NVDA,科技,220.78,7,100,90,100,95,★★★★★,強力買進,低點強買,持續改善,注意,穩健,384.511,611.588,1030.330,221.362,384.511,611.588,1030.330,1837.069,1.770,本益比,成長 → 成長 → 成長,成長 → 成長 → 成長,成長 → 成長 → 成長,33.810,14.424,6.530,15.306,27.358,0.780,0.763,0.008,0.198,0.195,0.220,0.358,0.354,0.204,0.391,0.524,0.546
4,NFLX,科技,81.05,5,100,95,100,97,★★★★★,強力買進,低點強買,表現分歧,注意,穩健,121.549,153.150,179.536,81.772,121.549,153.150,179.536,227.971,0.890,本益比,成長 → 成長 → 成長,衰退 → 成長 → 衰退,衰退 → 成長 → 衰退,25.511,21.212,3.177,3.821,11.193,0.312,0.413,0.075,0.025,0.017,0.047,-0.350,1.196,-0.046,-0.670,1.505,-0.253
5,BABA,科技,114.02,30,51,50,50,53,★★★☆☆,持有,買進,表現分歧,資料不足,穩健,113.262,122.538,136.045,93.928,113.262,122.538,136.045,164.723,0.075,本益比,成長 → 成長 → 衰退,衰退 → 衰退 → 成長,衰退 → 成長 → 衰退,2.649,12.228,43.040,9.324,1.707,0.051,0.098,-0.292,-0.146,0.150,0.001,0.757,-0.321,-0.516,-0.739,2.568,-0.511
6,ADBE,科技,292.79,14,85,90,100,89,★★★★★,強力買進,低點強買,表現分歧,良好,注意,495.093,645.276,813.803,251.058,495.093,645.276,813.803,961.770,1.204,本益比,成長 → 成長 → 成長,成長 → 成長 → 衰退,成長 → 衰退 → 衰退,16.750,10.648,17.480,27.498,10.143,0.533,0.613,0.079,0.034,0.033,0.034,-0.076,0.034,0.065,-0.268,-0.064,0.438
7,AMZN,科技,259.77,3,72,57,80,70,★★★★☆,買進,低點強買,資料不足,資料不足,穩健,434.029,512.177,756.081,354.745,434.029,512.177,756.081,900.185,0.972,本益比,成長 → 衰退 → 資料不足,持平 → 成長 → 成長,成長 → 衰退 → 資料不足,20.899,24.993,12.430,10.394,5.078,0.119,0.189,0.001,NaN,-0.149,0.184,1.068,0.426,0.000,NaN,-0.522,0.533
8,ORCL,科技,149.12,5,91,60,50,75,★★★★☆,買進,低點強買,持續改善,注意,注意,207.682,216.932,259.931,183.922,207.682,216.932,259.931,335.110,0.455,本益比,成長 → 成長 → 成長,成長 → 衰退 → 成長,衰退 → 成長 → 成長,25.578,13.648,5.830,10.926,11.436,0.101,0.402,-0.057,0.116,0.070,0.076,0.142,-0.395,1.079,1.044,2.461,-0.746
9,AAPL,科技,316.85,67,37,55,83,48,★★☆☆☆,減碼,持有,表現分歧,良好,資料不足,257.159,297.194,323.075,209.230,257.159,297.194,323.075,354.330,-0.062,本益比,成長 → 衰退 → 衰退,成長 → 衰退 → 成長,成長 → 衰退 → 成長,36.336,33.220,8.720,9.538,43.050,1.112,1.519,0.023,-0.016,-0.227,0.403,0.005,-0.292,0.535,0.197,-0.468,0.814


In [ ]:
# 選擇性歷史彙整：只讀指定版本，不會混合兩種欄位。
def collect_report_history(version="詳細版"):
    if version not in {"詳細版", "簡化版"}:
        raise ValueError("version must be 詳細版 or 簡化版")
    # ------------------------------------------------------------------
    # Cell — Historical Stock Report Collector (Enhanced)
    # ------------------------------------------------------------------
    from pathlib import Path
    from datetime import datetime
    import pandas as pd

    report_dir = Path("reports") / version

    # 只抓符合命名規則的歷史報表
    files = sorted(report_dir.rglob("US_Report_*.xlsx"))      

    all_reports = []

    for file in files:
        try:
            timestamp_str = file.stem.split("_Report_")[1]
            snapshot_time = datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S_%f" if timestamp_str.count("_") == 2 else "%Y%m%d_%H%M%S")

            df = pd.read_excel(file)
            if "投資建議" in df.columns and "基本面投資建議" not in df.columns:
                df = df.rename(columns={"投資建議": "基本面投資建議"})

            parents = file.parents
            config_name = parents[2].name if len(parents) >= 3 else "unknown"

            df.insert(0, "SnapshotTime", snapshot_time)
            df.insert(1, "Config", config_name)
            df.insert(2, "SourceFile", file.name)

            all_reports.append(df)
            print(f"Loaded: {file.name}")

        except Exception as e:
            print(f"Skip {file.name}: {e}")

    # -------------------------------------------------------------
    # 空資料保護
    # -------------------------------------------------------------
    if not all_reports:
        raise ValueError("No report files loaded. Please check the reports folder or file naming pattern.")

    # -------------------------------------------------------------
    # 合併全部歷史報表
    # -------------------------------------------------------------
    history_df = pd.concat(all_reports, ignore_index=True)
    history_df = history_df.sort_values("SnapshotTime").reset_index(drop=True)

    print(f"\nTotal rows: {len(history_df)}")
    if "代號" in history_df.columns:
        print(f"Total tickers: {history_df['代號'].nunique()}")

    # -------------------------------------------------------------
    # 匯出資料夾
    # -------------------------------------------------------------
    history_dir = report_dir / "history"
    history_dir.mkdir(parents=True, exist_ok=True)

    output_file = history_dir / f"WHID_US_History_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

    # -------------------------------------------------------------
    # 最新摘要
    # -------------------------------------------------------------
    if "代號" in history_df.columns:
        latest_df = (
            history_df
            .sort_values("SnapshotTime")
            .groupby("代號", as_index=False)
            .tail(1)
            .sort_values(["Config", "基本面分數"], ascending=[True, False], na_position="last")
            .reset_index(drop=True)
        )
    else:
        latest_df = history_df.copy()

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        # 1. 全部資料總表
        history_df.to_excel(writer, sheet_name="AllReports", index=False)

        # 2. 最新摘要
        latest_df.to_excel(writer, sheet_name="LatestSummary", index=False)

        # 3. 各股票分頁
        if "代號" in history_df.columns:
            tickers = history_df["代號"].dropna().unique()
            used_sheet_names = set(["AllReports", "LatestSummary"])

            for ticker in tickers:
                stock_df = history_df[history_df["代號"] == ticker].copy()
                stock_df = stock_df.sort_values("SnapshotTime").reset_index(drop=True)

                # -------------------------------------------------
                # 新增：現價溢價比例 = 現價 / 基本面買點
                # 新增：現價溢價比例變化
                # -------------------------------------------------
                if "現價" in stock_df.columns and "基本面買點" in stock_df.columns:
                    stock_df["現價溢價比例"] = stock_df["現價"] / stock_df["基本面買點"]
                    stock_df["現價溢價比例變化"] = stock_df["現價溢價比例"].diff()

                # -------------------------------------------------
                # 其他變化欄位
                # -------------------------------------------------
                if "現價" in stock_df.columns:
                    stock_df["現價變化"] = stock_df["現價"].diff()

                if "綜合分數" in stock_df.columns:
                    stock_df["基本面分數變化"] = stock_df["基本面分數"].diff()

                if "歷史估值百分位" in stock_df.columns:
                    stock_df["歷史估值百分位變化"] = stock_df["歷史估值百分位"].diff()

                if "基本面投資建議" in stock_df.columns:
                    stock_df["投資建議變化"] = stock_df["基本面投資建議"].shift(1).fillna("") + " → " + stock_df["基本面投資建議"]
                    stock_df.loc[stock_df.index == 0, "投資建議變化"] = stock_df.loc[stock_df.index == 0, "基本面投資建議"]

                # sheet name（代號即可，避免中文名稱管理）
                base_name = str(ticker)[:31]
                sheet_name = base_name
                counter = 1

                while sheet_name in used_sheet_names:
                    suffix = f"_{counter}"
                    sheet_name = base_name[:31 - len(suffix)] + suffix
                    counter += 1

                used_sheet_names.add(sheet_name)

                stock_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Saved: {output_file}")
    return output_file

# 需要時才手動呼叫：collect_report_history("詳細版")
